# 머신러닝 파일별 7 · 11 앙상블 ① Voting · Bagging · Random Forest

**한 줄 요약**

- 여러 모델을 합치면 언제 더 안정적이고, 언제 그렇지 않을까? · 트리의 흔들림 · Voting 반례 · Bagging · OOB
- Random Forest
- ExtraTrees를 같은 평가 계약으로 고르고 FINAL_HOLDOUT은 한 번만

- **원본** · `6.머닝러신/06_앙상블/11_ensemble.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 종합 정리 범위 밖 (파일별 6 결정 트리 → 파일별 7 → 보강 3 Boosting)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. bootstrap 표본만 바꿔도 단일 트리의 경계가 흔들리고 Bagging이 그 흔들림을 줄이는 것을 숫자로 보인다
2. Voting이 최고 단일 모델을 못 넘은 반례를 disagreement · error overlap으로 해석한다
3. Bagging → OOB → Random Forest를 '무엇을 다르게 보여 주나'의 순서로 설명한다
4. permutation importance · 피처 선택을 역할을 나눈 데이터로만 해석한다
5. RF와 ExtraTrees를 개발 데이터 CV로 고른 뒤 FINAL_HOLDOUT을 한 번만 연다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">트리 하나의 결과가 seed마다 달라질 때</td>
<td style="text-align: left;">Bagging · Random Forest (1 · 3장)</td>
</tr>
<tr>
<td style="text-align: left;">여러 모델을 합치면 무조건 좋아진다고 들었을 때</td>
<td style="text-align: left;">Voting 반례 · 오류 겹침 (2장)</td>
</tr>
<tr>
<td style="text-align: left;">검증 데이터를 따로 떼기 아까울 때</td>
<td style="text-align: left;">OOB 추정 · 단 CV · holdout의 대체가 아님 (3-1)</td>
</tr>
<tr>
<td style="text-align: left;">피처를 줄이려 할 때</td>
<td style="text-align: left;">permutation importance · 역할 분리 (4장)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/pfm7_map.png" alt="11 앙상블 ① 학습 지도" width="763">

**예상 학습 시간** · 개념 1시간 30분 + 실습 1시간 (전체 실행 3분 안팎 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">모델 · 평가 도구 · 원본 목표</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">트리의 흔들림</td>
<td style="text-align: left;">bootstrap 표본이 바뀌면 단일 트리와 Bagging의 경계는 얼마나 바뀌는가?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">Voting 반례</td>
<td style="text-align: left;">서로 다른 모델의 예측을 합치면 가장 좋은 단일 모델보다 좋아지는가?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">Bagging · OOB · Random Forest</td>
<td style="text-align: left;">같은 트리에 다른 표본(행)을 보여 주면, 그리고 다른 피처(열)까지 보여 주면 무엇이 달라지는가?</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">중요도와 피처 선택</td>
<td style="text-align: left;">permutation importance로 피처를 고르면 성능이 좋아지는가? 그 판단은 어느 데이터로 하는가?</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">RF vs ExtraTrees · 최종 1회</td>
<td style="text-align: left;">RF와 ExtraTrees 중 무엇을 고르고, FINAL_HOLDOUT은 언제 · 몇 번 여는가?</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">심화 · 이론은 경향</td>
<td style="text-align: left;">OOB는 언제 못 쓰고, RF와 ExtraTrees는 데이터 상황마다 승자가 정해져 있는가?</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">새 데이터에서 잠금 → 같은 CV로 후보 비교 → 고정 → holdout 1회를 처음부터 끝까지 하면?</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

원본의 plotly 그림 · `assets/` 슬라이드는 장 그림과 결과 줄로 바꿈. 원본에서 표가 Styler 객체로만 출력되던 셀은 숫자를 직접 출력함. **셀은 위에서부터 차례로 실행**.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;">numpy · pandas · scikit-learn (원본의 plotly 대신 정적 그림)</td>
</tr>
<tr>
<td style="text-align: left;">seed</td>
<td style="text-align: left;"><code>random_state=42</code> · CV = StratifiedKFold(5, shuffle=True, 42)</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;">가짜 2차원 260개 · Wine 178개(개발 142 · FINAL_HOLDOUT 36) · 가짜 시나리오 데이터 · 종합 실습은 유방암 569개</td>
</tr>
<tr>
<td style="text-align: left;">원본 그림</td>
<td style="text-align: left;"><code>assets/</code> 폴더 슬라이드 12장 · 첨부 이미지 1장 → 원본 폴더에 없어 깨진 상태 → 💡 · 비교 표의 글로</td>
</tr>
</tbody>
</table>

In [1]:
# 준비: 이 노트북에서 쓰는 모델 · 평가 도구 (원본의 plotly 그림은 images/의 정적 그림으로)
import numpy as np                                                     # 배열
import pandas as pd                                                    # 표
from sklearn.datasets import load_wine, make_classification            # Wine · 가짜 데이터
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier   # 배깅 · 엑스트라 트리
from sklearn.ensemble import RandomForestClassifier, VotingClassifier  # 랜덤 포레스트 · 투표
from sklearn.linear_model import LogisticRegression                    # 로지스틱
from sklearn.metrics import accuracy_score                             # 정확도
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score  # CV
from sklearn.model_selection import train_test_split                   # 분할
from sklearn.neighbors import KNeighborsClassifier                     # KNN
from sklearn.pipeline import Pipeline                                  # 전처리 + 모델
from sklearn.preprocessing import StandardScaler                       # 스케일링
from sklearn.tree import DecisionTreeClassifier as DT                  # 결정 트리 (짧은 이름)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 오늘의 CV 계약
print("준비 완료 · CV = StratifiedKFold(5, shuffle=True, 42)")   # 확인

준비 완료 · CV = StratifiedKFold(5, shuffle=True, 42)


### 원본 목표 · 오늘의 관찰 문제

- **오늘의 문제** · 여러 모델을 합치면 언제 더 안정적이고, 언제 그렇지 않을까?
- 목표 · 트리 경계의 흔들림 관찰 · Voting 반례 해석 · Bagging → OOB → Random Forest · RF vs ExtraTrees 선택 후 FINAL_HOLDOUT 1회

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">결정 트리 장점</th>
<th style="text-align: left;">오늘 확인할 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">사람이 규칙을 읽을 수 있다</td>
<td style="text-align: left;">학습 표본이 조금 바뀌면 경계도 많이 바뀔까?</td>
</tr>
<tr>
<td style="text-align: left;">스케일링이 필수가 아님</td>
<td style="text-align: left;">여러 Tree의 표를 모으면 흔들림은 어떻게 달라질까?</td>
</tr>
</tbody>
</table>

**원본 힌트**

- 원본 지도 · Voting = 서로 다른 모델 결합 · Bagging = 같은 모델에 다른 표본 · RF = Bagging + feature 무작위 · Boosting = 순차 (보강 3)
- Core 문장 하나 · Variance가 높으면 학습 표본이 조금 바뀔 때 예측이 크게 흔들릴 수 있다

---
## 1. 트리의 흔들림

> **이 장의 흐름**
>
> - 같은 데이터에서 bootstrap 표본 3개 → 트리 · Bagging 경계 불일치율
> - 그다음 Bias vs Variance 곡선으로 '흔들림 = Variance'에 이름 붙이기

<img src="images/pfm7_boundary.png" alt="bootstrap 표본 3개 · 트리(위)와 Bagging(아래)의 경계" width="912">

**파이프라인 위치** · ④ 모델 — 분산(Variance)

### 1-1. `A` 표본을 바꾸면 경계가 흔들림 · `bootstrap` · `BaggingClassifier` <sub>원본 셀 2 ~ 4</sub>

> 🎯 **bootstrap 표본만 바꿨을 때 트리와 Bagging 중 어느 쪽 경계가 더 흔들리는가?**

💡 **쉽게 말하면**

- **bootstrap** = 같은 개수만큼 **복원추출** → 어떤 점은 두 번, 어떤 점은 빠짐.
- 트리는 표본에 예민 · Bagging은 트리 80그루의 표를 모아 평균.

**실행 흐름**

1. 가짜 2차원 260개
2. seed 3 · 17 · 91로 bootstrap
3. 트리 · Bagging 학습 → 격자 예측
4. 세 경계끼리 다른 칸 비율

**먼저 예측** · 경계 불일치율은 트리와 Bagging 중 어느 쪽이 클까? — 답은 코드 실행 결과 아래에

In [2]:
X_ins, y_ins = make_classification(n_samples=260, n_features=2, n_redundant=0,   # 원본 셀 3
                                   n_informative=2, n_clusters_per_class=1, class_sep=0.9,
                                   flip_y=0.08, random_state=42)
x0 = np.linspace(X_ins[:, 0].min() - 0.7, X_ins[:, 0].max() + 0.7, 130)          # 격자 가로
x1 = np.linspace(X_ins[:, 1].min() - 0.7, X_ins[:, 1].max() + 0.7, 130)          # 격자 세로
grid_points = np.c_[[a.ravel() for a in np.meshgrid(x0, x1)]].T                  # 130 × 130 칸
tree_b, bag_b = [], []                                                           # 표본마다 경계
for seed in [3, 17, 91]:                                                         # bootstrap 3번
    idx = np.random.default_rng(seed).choice(260, size=260, replace=True)        # 복원추출
    Xb, yb = X_ins[idx], y_ins[idx]                                              # 이번 표본
    tree_b.append(DT(random_state=seed).fit(Xb, yb).predict(grid_points))        # 트리 경계
    bag = BaggingClassifier(estimator=DT(), n_estimators=80, random_state=seed)  # 80그루
    bag_b.append(bag.fit(Xb, yb).predict(grid_points))                           # Bagging 경계


def mean_pair_disagreement(bs):  # 세 경계끼리 다른 칸 비율의 평균
    return np.mean([np.mean(bs[i] != bs[j]) for i in range(3) for j in range(i + 1, 3)])  # 쌍 3개


print(f"Single Tree 경계 불일치율: {mean_pair_disagreement(tree_b):.3f}")  # 원본은 Styler로만 출력
print(f"Bagging     경계 불일치율: {mean_pair_disagreement(bag_b):.3f}")   # 흔들림 비교

Single Tree 경계 불일치율: 0.108
Bagging     경계 불일치율: 0.061


**결과** · 예측 정답은 트리

- `Single Tree 경계 불일치율: 0.108` — 표본만 바뀌어도 격자의 11%가 다른 답
- `Bagging     경계 불일치율: 0.061` — 80그루 평균 → 흔들림이 약 0.56배로 줄어듦

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">단일 트리</td>
<td style="text-align: left;">표본에 예민 · 경계가 크게 바뀜</td>
<td style="text-align: left;">규칙을 읽어야 할 때</td>
</tr>
<tr>
<td style="text-align: left;">Bagging</td>
<td style="text-align: left;">같은 트리에 다른 표본 → 평균</td>
<td style="text-align: left;">흔들림을 줄이고 싶을 때</td>
</tr>
<tr>
<td style="text-align: left;">경계 불일치율</td>
<td style="text-align: left;">두 경계가 다르게 답하는 칸의 비율</td>
<td style="text-align: left;">흔들림을 숫자로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 3은 표를 Styler 객체로만 출력해 숫자가 화면에 없었음 → 여기서는 숫자로 출력
> - 이 값은 이 데이터 · 이 seed 3개의 결과 → 정확한 배율을 일반화하지 않기

> ✅ **한 줄 정리** · bootstrap 표본만 바꿔도 단일 트리의 경계는 크게 흔들리고, Bagging은 여러 트리의 평균으로 그 흔들림을 줄인다.

**딥러닝 코드에서는**

- seed만 바꿔 학습해도 예측이 달라짐 → 모델 여러 개 평균(앙상블)이 안정적

### 1-2. `B` Bias vs Variance · `bias²` · `variance` <sub>원본 셀 5 ~ 11</sub>

> 🎯 **모델이 복잡해지면 편향과 분산은 각각 어떻게 움직이고, 총 오차는 어디서 가장 작은가?**

💡 **쉽게 말하면**

- **Bias** = 중요한 패턴을 충분히 못 배우는 경향 (과소적합).
- **Variance** = 학습 데이터 변화에 예측이 민감한 경향 (과적합).

**실행 흐름**

1. 복잡도 0 ~ 10
2. 원본 식으로 bias² · variance · 합
3. 합이 가장 작은 복잡도

**먼저 예측** · 총 오차가 가장 작은 복잡도는? — 답은 코드 실행 결과 아래에

In [3]:
complexity = np.linspace(0, 10, 101)              # 모델 복잡도 (원본 셀 9의 장난감 식)
bias2 = (10 - complexity) ** 2 / 10               # 복잡할수록 줄어듦
variance = complexity ** 2 / 10                   # 복잡할수록 커짐
error = bias2 + variance                          # 총 오차
best = complexity[np.argmin(error)]               # 균형점
print(f"균형점 복잡도 {best:.1f} · 총 오차 {error.min():.1f}")  # 원본 그림의 초록 점선
for c in [0, 5, 10]:                              # 세 지점
    i = int(c * 10)                               # 배열 위치
    b2, va, er = bias2[i], variance[i], error[i]  # 세 값
    print(f"복잡도 {c:>2}: bias² {b2:.1f} · variance {va:.1f} · 합 {er:.1f}")  # 표

균형점 복잡도 5.0 · 총 오차 5.0
복잡도  0: bias² 10.0 · variance 0.0 · 합 10.0
복잡도  5: bias² 2.5 · variance 2.5 · 합 5.0
복잡도 10: bias² 0.0 · variance 10.0 · 합 10.0


**결과** · 예측 정답은 5 (균형점)

- `균형점 복잡도 5.0 · 총 오차 5.0` — 원본 그림의 '균형점' 점선 위치
- `복잡도  0: bias² 10.0 · variance 0.0 · 합 10.0` — 너무 단순 → 편향이 큼 (과소적합)
- `복잡도 10: bias² 0.0 · variance 10.0 · 합 10.0` — 너무 복잡 → 분산이 큼 (과적합)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Low Bias · Low Var</td>
<td style="text-align: left;">이상적 · 중심 근처에 모인 점</td>
<td style="text-align: left;">목표</td>
</tr>
<tr>
<td style="text-align: left;">Low Bias · High Var</td>
<td style="text-align: left;">과적합 · 중심 주변 흩어짐</td>
<td style="text-align: left;">깊은 트리 → Bagging · RF</td>
</tr>
<tr>
<td style="text-align: left;">High Bias · Low Var</td>
<td style="text-align: left;">과소적합 · 한쪽에 몰림</td>
<td style="text-align: left;">얕은 모델 → Boosting (보강 3)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 이 곡선은 개념을 보여 주는 장난감 식 → 실제 모델의 오차는 데이터로 확인
> - 원본의 Bias · Variance 과녁 그림(첨부 이미지)은 노트북에 들어 있지 않아 이 표로 대신함

> ✅ **한 줄 정리** · 복잡도가 커지면 편향은 줄고 분산은 커지며, Bagging은 분산을, Boosting은 편향을 줄이려는 방법이다.

**딥러닝 코드에서는**

- 큰 신경망 = 낮은 bias · 높은 variance → dropout · 앙상블 · 데이터 증강으로 분산을 줄임

---
## 2. Voting 반례

> **이 장의 흐름**
>
> - Wine을 개발 142 · FINAL_HOLDOUT 36으로 잠금 → 로지스틱 · KNN · 트리 · Hard · Soft Voting을 같은 CV로
> - 그다음 모델끼리 예측이 얼마나 다르고 오류가 얼마나 겹치는지

<img src="images/pfm7_voting.png" alt="Wine 개발 CV · 단일 모델 3개와 Voting 2개" width="636">

**파이프라인 위치** · ⑤ 교차검증 — 모델 결합

### 2-1. `A` Wine 잠금 · 기본 모델 셋 · `open_final_holdout_once` · `Pipeline` <sub>원본 셀 16 ~ 18</sub>

> 🎯 **선택을 끝내기 전에는 열 수 없는 FINAL_HOLDOUT과 기본 모델 셋은 어떻게 준비하는가?**

💡 **쉽게 말하면**

- **평가 계약** · 먼저 FINAL_HOLDOUT을 잠그고, 모든 선택은 개발 데이터(DEVELOPMENT_POOL) 안에서.
- 로지스틱 · KNN은 스케일러를 붙이고, 트리는 그대로.

**실행 흐름**

1. Wine 178 → 개발 142 · 잠금 36
2. 한 번만 여는 함수
3. 모델 셋 정의

**먼저 예측** · 모델 선택 전에 holdout을 열면? — 답은 코드 실행 결과 아래에

In [4]:
wine = load_wine()                                         # 와인 178병 · 13특성 · 3품종
X_development, X_final_holdout, y_development, y_final_holdout = train_test_split(  # 먼저 잠금
    wine.data, wine.target, test_size=0.2, stratify=wine.target, random_state=42)
Xd, yd = X_development, y_development                      # 짧은 이름
FINAL_HOLDOUT_ACCESS_COUNT, FINAL_MODEL_FROZEN = 0, False  # 연 횟수 · 선택 종료 여부


def open_final_holdout_once():                                   # 원본 셀 17
    global FINAL_HOLDOUT_ACCESS_COUNT                            # 바깥 변수를 바꿈
    if not FINAL_MODEL_FROZEN:                                   # 선택이 안 끝났으면
        # 막음
        raise RuntimeError("모델과 하이퍼파라미터 선택을 먼저 완료한 뒤 FINAL_HOLDOUT을 열기")
    if FINAL_HOLDOUT_ACCESS_COUNT != 0:                          # 이미 열었으면
        raise RuntimeError("FINAL_HOLDOUT은 이미 한 번 열었음")  # 막음
    FINAL_HOLDOUT_ACCESS_COUNT += 1                              # 기록
    return X_final_holdout, y_final_holdout                      # 꺼내 줌


print("DEVELOPMENT_POOL:", Xd.shape, "· FINAL_HOLDOUT (locked):", X_final_holdout.shape)  # 142 · 36
try:                                                                 # 선택 전에 열어 보기
    open_final_holdout_once()                                        # 아직 FROZEN 아님
except RuntimeError as e:                                            # 막힘
    print("열기 시도:", e)                                           # 안내
lr5000 = LogisticRegression(max_iter=5000, random_state=42)          # 원본 설정
log_clf = Pipeline([("scaler", StandardScaler()), ("clf", lr5000)])  # 스케일러 + 로지스틱
# 이웃 7
knn_clf = Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=7))])
dt_clf = DT(max_depth=4, random_state=42)                            # 깊이 4 트리

DEVELOPMENT_POOL: (142, 13) · FINAL_HOLDOUT (locked): (36, 13)
열기 시도: 모델과 하이퍼파라미터 선택을 먼저 완료한 뒤 FINAL_HOLDOUT을 열기


**결과** · 예측 정답은 RuntimeError — 선택을 먼저 완료

- `DEVELOPMENT_POOL: (142, 13) · FINAL_HOLDOUT (locked): (36, 13)` — 개발 142 · 잠금 36
- `열기 시도: 모델과 하이퍼파라미터 선택을 먼저 완료한 뒤 FINAL_HOLDOUT을 열기` — 선택 전에는 함수가 막아 줌

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">DEVELOPMENT_POOL 142</td>
<td style="text-align: left;">모델 · 하이퍼파라미터 · family 선택</td>
<td style="text-align: left;">여러 번</td>
</tr>
<tr>
<td style="text-align: left;">FINAL_HOLDOUT 36</td>
<td style="text-align: left;">선택 종료 뒤 1회</td>
<td style="text-align: left;">5-2</td>
</tr>
<tr>
<td style="text-align: left;">스케일러 + 로지스틱 · KNN</td>
<td style="text-align: left;">거리 · 선형은 단위를 맞춤</td>
<td style="text-align: left;">트리는 그대로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본은 `X_train, y_train = X_development, y_development` 호환 이름도 둠 → 둘 다 개발 데이터 (holdout 아님)

> ✅ **한 줄 정리** · FINAL_HOLDOUT은 처음에 잠그고 선택이 끝난 뒤에만 한 번 열 수 있게 만들며, 모든 비교는 개발 데이터 안에서 한다.

**딥러닝 코드에서는**

- test loader를 함수로 감싸 한 번만 부르게 만드는 습관

### 2-2. `A` Hard · Soft Voting CV 비교 · `VotingClassifier` · `voting="soft"` <sub>원본 셀 12 ~ 21</sub>

> 🎯 **Hard Voting(라벨 다수결)과 Soft Voting(확률 평균)은 가장 좋은 단일 모델을 넘는가?**

💡 **쉽게 말하면**

- **Hard** = 라벨 다수결 · **Soft** = 예측 확률 평균.
- Soft는 확률 품질 · 모델 다양성에 따라 결과가 달라져 항상 우월하지 않음.

**실행 흐름**

1. Voting 두 개 정의
2. 모델 5개를 같은 CV로
3. 평균 · 표준편차 · OOF 예측 저장

**먼저 예측** · Voting이 1등 단일 모델보다 높을까? — 답은 코드 실행 결과 아래에

In [5]:
members = [("lr", log_clf), ("knn", knn_clf), ("dt", dt_clf)]                   # 서로 다른 세 모델
models = {"Logistic": log_clf, "KNN": knn_clf, "Decision Tree": dt_clf,         # 단일 3개
          "Hard Voting": VotingClassifier(members, voting="hard"),              # 라벨 다수결
          "Soft Voting": VotingClassifier(members, voting="soft")}              # 확률 평균
rows, oof_predictions = [], {}                                                  # 표 · fold 밖 예측
for name, model in models.items():                                              # 같은 CV 계약
    s = cross_val_score(model, Xd, yd, cv=cv, scoring="accuracy", n_jobs=-1)    # 정확도 5개
    oof_predictions[name] = cross_val_predict(model, Xd, yd, cv=cv, n_jobs=-1)  # 2-3에서 씀
    rows.append([name, s.mean(), s.std()])                                      # 한 줄
voting_cv = pd.DataFrame(rows, columns=["Model", "CV mean", "CV std"])          # 표
voting_cv = voting_cv.sort_values("CV mean", ascending=False)                   # 1등부터
print(voting_cv.round(3).to_string(index=False))  # 원본은 Styler로만 출력

        Model  CV mean  CV std
     Logistic    0.979   0.028
  Hard Voting    0.972   0.040
  Soft Voting    0.965   0.031
          KNN    0.958   0.041
Decision Tree    0.902   0.059


**결과** · 예측 정답은 아님 — Logistic 0.979 > Hard 0.972 > Soft 0.965

- `Logistic    0.979   0.028` — 1등 단일 모델
- `Hard Voting    0.972   0.040` — 합쳤는데 1등보다 낮음
- `Soft Voting    0.965   0.031` — 확률 평균도 넘지 못함 · KNN 0.958 · 트리 0.902

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Hard Voting</td>
<td style="text-align: left;">라벨 다수결</td>
<td style="text-align: left;">확률이 없는 모델이 섞였을 때</td>
</tr>
<tr>
<td style="text-align: left;">Soft Voting</td>
<td style="text-align: left;">확률 평균 · 확신도 반영</td>
<td style="text-align: left;">확률 품질이 비슷하게 좋을 때</td>
</tr>
<tr>
<td style="text-align: left;">최고 단일 모델</td>
<td style="text-align: left;">이 조합의 기준선</td>
<td style="text-align: left;">Voting이 넘는지 먼저 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 결론은 'Voting은 나쁘다'가 아니라 '**이 조합에서는 최고 단일 모델보다 개선되지 않았다**'로 제한
> - 점수가 예상과 달라도 seed를 바꾸거나 반례를 숨기지 않음
> - 확률 보정(calibration)은 별도 개발 근거로 확인 · 이 반례를 뒤집기 위한 사후 조치가 아님

> ✅ **한 줄 정리** · 서로 다른 모델을 합친 Voting도 이 조합에서는 최고 단일 모델(0.979)을 넘지 못했으며, 그 반례를 그대로 기록한다.

**딥러닝 코드에서는**

- 체크포인트 · 모델 앙상블도 구성원이 약하거나 비슷하면 효과가 작음

### 2-3. `A` disagreement와 error overlap · `oof_predictions` · `Jaccard` <sub>원본 셀 21 ~ 25</sub>

> 🎯 **합쳐도 개선되지 않은 이유를 알려면 모델들이 얼마나 다르게 예측하는지만 보면 충분한가?**

💡 **쉽게 말하면**

- 예측이 다르다는 사실만으로 좋은 앙상블이 아님.
- 구성원의 기본 품질과 **오류가 어떻게 겹치는지**를 함께 봄.

**실행 흐름**

1. 쌍마다 예측이 다른 비율
2. 둘 다 틀린 비율
3. 오류 겹침(Jaccard)
4. 최고 단일 vs 최고 Voting

**먼저 예측** · 예측이 가장 많이 다른 쌍이 가장 좋은 조합일까? — 답은 코드 실행 결과 아래에

In [6]:
base = ["Logistic", "KNN", "Decision Tree"]                           # 단일 모델 3개
for i, a in enumerate(base):                                          # 쌍마다
    for b in base[i + 1:]:                                            # 중복 없이
        ea, eb = oof_predictions[a] != yd, oof_predictions[b] != yd   # 각자 틀린 자리
        disagree = np.mean(oof_predictions[a] != oof_predictions[b])  # 예측이 다른 비율
        union = (ea | eb).sum()                                       # 한쪽이라도 틀린 수
        jac = (ea & eb).sum() / union if union else 0.0               # 오류 겹침
        both = np.mean(ea & eb)                                       # 둘 다 틀린 비율
        print(f"{a} vs {b}: disagreement {disagree:.3f} · "           # 쌍마다 세 지표
              f"both wrong {both:.3f} · overlap {jac:.3f}")
comparison = voting_cv.set_index("Model")["CV mean"]                  # 2-2 표
best_single = comparison.loc[base].max()                              # 단일 모델 최고
best_voting = comparison.loc[["Hard Voting", "Soft Voting"]].max()    # Voting 최고
print(f"Best single CV: {best_single:.3f} · Best voting CV: {best_voting:.3f}")  # 원본 셀 22

Logistic vs KNN: disagreement 0.028 · both wrong 0.021 · overlap 0.500
Logistic vs Decision Tree: disagreement 0.106 · both wrong 0.007 · overlap 0.062
KNN vs Decision Tree: disagreement 0.113 · both wrong 0.014 · overlap 0.111
Best single CV: 0.979 · Best voting CV: 0.972


**결과** · 예측 정답은 아님 — 트리와의 쌍이 많이 다르지만 트리 자체가 약함

- `Logistic vs KNN: disagreement 0.028 · both wrong 0.021 · overlap 0.500` — 둘이 거의 같은 답 · 틀린 자리의 절반이 겹침
- `Logistic vs Decision Tree: disagreement 0.106 · both wrong 0.007 · overlap 0.062` — 많이 다르지만 트리 자체가 많이 틀림 (CV 0.902)
- `Best single CV: 0.979 · Best voting CV: 0.972` — 이 조합에서 Voting이 최강 단일 모델을 넘지 못함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Prediction disagreement</td>
<td style="text-align: left;">두 모델 예측이 다른 비율</td>
<td style="text-align: left;">다양성의 한 면</td>
</tr>
<tr>
<td style="text-align: left;">Both wrong</td>
<td style="text-align: left;">둘 다 틀린 비율</td>
<td style="text-align: left;">같이 틀리면 다수결도 틀림</td>
</tr>
<tr>
<td style="text-align: left;">Error overlap (Jaccard)</td>
<td style="text-align: left;">틀린 자리의 겹침 정도</td>
<td style="text-align: left;">구성원 품질과 함께 읽기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 완전한 독립을 가정하지 않음 · 비슷하게 움직일수록 평균 효과가 줄 수 있다는 직관만
> - Jaccard 식 계산은 선택 · Core에서는 '오류가 함께 겹치는 정도'로 읽음

> ✅ **한 줄 정리** · 앙상블 효과는 예측이 다른 정도만이 아니라 구성원 품질과 오류가 겹치는 정도에 달려 있다.

**딥러닝 코드에서는**

- 같은 구조 · 같은 데이터로 학습한 모델끼리는 오류가 겹쳐 앙상블 효과가 작음

---
## 3. Bagging · OOB · Random Forest

> **이 장의 흐름**
>
> - Bagging과 OOB(빠진 표본으로 내부 추정) → 2차원 데모로 트리 vs Bagging
> - 그다음 max_features로 피처까지 무작위 = Random Forest → Wine에서 셋 비교

<img src="images/pfm7_oob.png" alt="bootstrap에서 빠진 약 37% = OOB" width="763">

**파이프라인 위치** · ④ 모델 → ⑤ 교차검증

### 3-1. `A` Bagging과 OOB · `oob_score=True` · `oob_score_` <sub>원본 셀 26 ~ 30</sub>

> 🎯 **한 샘플을 평가할 때 그 샘플을 학습에서 보지 않은 트리들의 표만 모으면 무엇이 되는가?**

💡 **쉽게 말하면**

- bootstrap에서 **빠진 표본**(Out-of-Bag)은 그 트리에게 처음 보는 문제.
- OOB = 개발 데이터 **내부의 보조 추정치** · FINAL_HOLDOUT이 아님.

**실행 흐름**

1. 300그루 Bagging · oob_score=True
2. 같은 개발 데이터로 CV
3. fit 후 OOB 점수 · 빠진 비율

**먼저 예측** · CV와 OOB 값이 같으면 같은 평가법일까? — 답은 코드 실행 결과 아래에

In [7]:
bag_wine = BaggingClassifier(estimator=DT(random_state=42), n_estimators=300,  # 원본 셀 29
                             bootstrap=True, oob_score=True, n_jobs=-1, random_state=42)
bag_scores = cross_val_score(bag_wine, Xd, yd, cv=cv, scoring="accuracy", n_jobs=-1)  # CV
bag_wine.fit(Xd, yd)                                        # 개발 142개 전체로 fit → OOB 계산
print(f"Development CV: {bag_scores.mean():.3f} ± {bag_scores.std():.3f}")  # fold별 평균 · 흔들림
print(f"OOB estimate  : {bag_wine.oob_score_:.3f}")         # 빠진 표본의 표만 모은 추정
in_bag = np.zeros(len(yd))                                  # 샘플마다 안 빠진 횟수
for s in bag_wine.estimators_samples_:                      # 그루마다 학습에 쓴 번호
    in_bag[np.unique(s)] += 1                               # 쓴 샘플 표시
oob_rate = 1 - in_bag.mean() / 300                          # 그루당 빠진 비율
print(f"트리 한 그루에서 빠진 비율 평균: {oob_rate:.3f} (이론값 1 - 1/e ≈ 0.368)")  # 약 37%
print("FINAL_HOLDOUT access:", FINAL_HOLDOUT_ACCESS_COUNT)  # 0

Development CV: 0.944 ± 0.041
OOB estimate  : 0.944
트리 한 그루에서 빠진 비율 평균: 0.366 (이론값 1 - 1/e ≈ 0.368)
FINAL_HOLDOUT access: 0


**결과** · 예측 정답은 아님 — 숫자가 같아도 만들어지는 과정이 다름

- `Development CV: 0.944 ± 0.041` — 5-Fold 평균 · 흔들림까지
- `OOB estimate  : 0.944` — 오늘은 CV와 같은 값
- `트리 한 그루에서 빠진 비율 평균: 0.366 (이론값 1 - 1/e ≈ 0.368)` — 그루마다 약 37%가 OOB
- `FINAL_HOLDOUT access: 0` — 여전히 잠김

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">CV</td>
<td style="text-align: left;">fold별 mean · std 직접 제공</td>
<td style="text-align: left;">모델 선택의 기본</td>
</tr>
<tr>
<td style="text-align: left;">OOB</td>
<td style="text-align: left;">bootstrap에서 빠진 표본으로 한 번에 추정</td>
<td style="text-align: left;">빠른 보조 추정 (Bagging · RF)</td>
</tr>
<tr>
<td style="text-align: left;">FINAL_HOLDOUT</td>
<td style="text-align: left;">선택 종료 뒤 1회</td>
<td style="text-align: left;">보고</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - OOB는 FINAL_HOLDOUT이 아니며, 모든 상황에서 CV를 대체한다는 규칙도 아님
> - 숫자 하나보다 평가 역할과 생성 과정을 먼저 구분

> ✅ **한 줄 정리** · OOB는 bootstrap에서 빠진 샘플을 그 샘플을 안 본 트리들로 평가한 내부 추정치이며, CV · holdout과 역할이 다르다.

**딥러닝 코드에서는**

- 학습 중 val로 쓰는 MC dropout · 앙상블 불확실성 추정과 비슷한 '내부 추정'

### 3-2. `B` 2차원 데모 · 트리 vs Bagging · `max_samples=0.8` <sub>원본 셀 39 ~ 42</sub>

> 🎯 **500개 2차원 데이터에서 트리와 Bagging의 검증 · CV 정확도는?**

💡 **쉽게 말하면**

- 1-1에서 본 흔들림을 점수로 한 번 더.
- Bagging은 표본의 80%씩 보여 준 트리 100그루.

**실행 흐름**

1. 500개 · 7 : 3 분할
2. 트리 · Bagging 검증 정확도
3. 같은 CV 평균 · 표준편차

**먼저 예측** · Bagging이 트리보다 높을까? — 답은 코드 실행 결과 아래에

In [8]:
X2, y2 = make_classification(  # 원본 셀 41 · 2차원 500개
    n_samples=500, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.0, random_state=42)
Xa, Xb, ya, yb = train_test_split(X2, y2, test_size=0.3, stratify=y2, random_state=42)  # 데모 분할


def make_bag():  # 원본 설정의 Bagging
    return BaggingClassifier(DT(), n_estimators=100, max_samples=0.8, random_state=42)  # 80%씩


tree_acc = accuracy_score(yb, DT(random_state=42).fit(Xa, ya).predict(Xb))          # 트리
bag_acc = accuracy_score(yb, make_bag().fit(Xa, ya).predict(Xb))                    # Bagging
print(f"Single Tree Demo Validation Acc: {tree_acc:.4f} · Bagging: {bag_acc:.4f}")  # 원본 셀 42
tree_cv = cross_val_score(DT(random_state=42), X2, y2, cv=cv, n_jobs=-1)            # 같은 CV
bag_cv = cross_val_score(make_bag(), X2, y2, cv=cv, n_jobs=-1)                      # 같은 CV
print(f"Single Tree Demo CV: {tree_cv.mean():.3f} ± {tree_cv.std():.3f}")           # 평균 · 흔들림
print(f"Bagging     Demo CV: {bag_cv.mean():.3f} ± {bag_cv.std():.3f}")             # 평균 · 흔들림

Single Tree Demo Validation Acc: 0.9400 · Bagging: 0.9667


Single Tree Demo CV: 0.944 ± 0.022
Bagging     Demo CV: 0.952 ± 0.021


**결과** · 예측 정답은 높음 — 0.944 → 0.952 (CV)

- `Single Tree Demo Validation Acc: 0.9400 · Bagging: 0.9667` — 검증에서 4개 차이
- `Single Tree Demo CV: 0.944 ± 0.022` — 트리
- `Bagging     Demo CV: 0.952 ± 0.021` — 평균은 조금 오르고 흔들림은 비슷

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">검증 한 번</td>
<td style="text-align: left;">150개 · 운이 섞임</td>
<td style="text-align: left;">빠른 확인</td>
</tr>
<tr>
<td style="text-align: left;">CV 평균 ± 표준편차</td>
<td style="text-align: left;">500개 전부 한 번씩 검증</td>
<td style="text-align: left;">비교의 기본</td>
</tr>
<tr>
<td style="text-align: left;">경계 그림 (1장)</td>
<td style="text-align: left;">모양으로 흔들림을 봄</td>
<td style="text-align: left;">직관</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - Core 비교는 3-3의 Wine · 이 데모는 시간이 있을 때만 보는 선택 실험

> ✅ **한 줄 정리** · 같은 2차원 데이터에서 Bagging은 트리보다 CV가 조금 높았지만, 차이는 fold 흔들림과 함께 읽는다.

**딥러닝 코드에서는**

- 데이터 증강 = 같은 모델에 '조금 다른 표본'을 보여 주는 같은 생각

### 3-3. `A` Random Forest · Wine에서 셋 비교 · `max_features="sqrt"` · `RandomForestClassifier` <sub>원본 셀 31 ~ 38 · 43 · 44</sub>

> 🎯 **행(표본)뿐 아니라 피처 후보까지 무작위로 하면 Wine CV는 어떻게 되는가?**

💡 **쉽게 말하면**

- 강한 피처가 늘 먼저 보이면 bootstrap이 달라도 트리들이 **닮음** → 평균 효과 ↓.
- **RF = Bagging + 분기마다 피처 후보 제한(`max_features`)** → 트리들이 덜 닮게.

**실행 흐름**

1. Single Tree · Bagging · RF 300그루
2. 같은 CV 평균 · 표준편차
3. 트리끼리 첫 질문이 얼마나 다른지

**먼저 예측** · 평균 순위와 흔들림(std) 순위는? — 답은 코드 실행 결과 아래에

In [9]:
core_models = {"Single Tree": DT(random_state=42),                        # 원본 셀 44
               "Bagging": BaggingClassifier(estimator=DT(random_state=42), n_estimators=300,
                                            n_jobs=-1, random_state=42),  # 행만 무작위
               "Random Forest": RandomForestClassifier(n_estimators=300, max_features="sqrt",
                                                       # 행 + 열 무작위
                                                       n_jobs=-1, random_state=42)}
for name, model in core_models.items():                                   # 같은 CV
    s = cross_val_score(model, Xd, yd, cv=cv, scoring="accuracy", n_jobs=-1)  # 정확도 5개
    print(f"{name:<14s} CV {s.mean():.3f} ± {s.std():.3f}")               # 원본은 Styler로만 출력
for name in ["Bagging", "Random Forest"]:                                 # 트리들이 얼마나 닮았나
    m = core_models[name].fit(Xd, yd)                                     # 개발 데이터로 fit
    roots = [est.tree_.feature[0] for est in m.estimators_]               # 그루마다 첫 질문 피처
    top_share = np.bincount(roots).max() / 300                            # 가장 많은 첫 질문의 비율
    # 닮은 정도
    print(f"{name}: 첫 질문 피처 종류 {len(set(roots))}개 · 최다 피처 비율 {top_share:.2f}")

Single Tree    CV 0.902 ± 0.059


Bagging        CV 0.944 ± 0.041


Random Forest  CV 0.979 ± 0.028
Bagging: 첫 질문 피처 종류 6개 · 최다 피처 비율 0.63


Random Forest: 첫 질문 피처 종류 11개 · 최다 피처 비율 0.21


**결과** · 예측 정답은 RF 1등 · 흔들림도 가장 작음

- `Single Tree    CV 0.902 ± 0.059` — 가장 낮고 가장 흔들림
- `Bagging        CV 0.944 ± 0.041` — 평균으로 크게 오름
- `Random Forest  CV 0.979 ± 0.028` — 피처까지 무작위 → 가장 높고 가장 안정
- `Bagging: 첫 질문 피처 종류 6개 · 최다 피처 비율 0.63` — Bagging 트리 63%가 같은 첫 질문
- `Random Forest: 첫 질문 피처 종류 11개 · 최다 피처 비율 0.21` — RF는 11가지로 흩어짐 = 덜 닮음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>n_estimators</code></td>
<td style="text-align: left;">트리 개수 · 많을수록 평균 효과 ↑ · 시간 · 메모리 ↑</td>
<td style="text-align: left;">충분히 크게 (수백)</td>
</tr>
<tr>
<td style="text-align: left;"><code>max_features</code></td>
<td style="text-align: left;">분기마다 고려할 피처 수 · 작을수록 다양성 ↑ · 너무 작으면 정보 손실</td>
<td style="text-align: left;">분류 기본 sqrt</td>
</tr>
<tr>
<td style="text-align: left;"><code>max_depth</code> · <code>min_samples_leaf</code></td>
<td style="text-align: left;">트리 복잡도</td>
<td style="text-align: left;">개발 데이터 CV로 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - RF가 Bagging보다 항상 높은 것은 아님 → 이 Wine 실행으로만 해석
> - 피처 무작위는 상관과 분산을 줄일 수 있지만 정보 손실 · bias 증가도 가능

> ✅ **한 줄 정리** · Random Forest는 Bagging에 피처 후보 무작위를 더해 트리들을 덜 닮게 만들고, 그 효과는 같은 CV의 평균 · 표준편차로 확인한다.

**딥러닝 코드에서는**

- dropout = 매 단계 일부 뉴런을 빼서 서로 덜 의존하게 만드는 같은 생각

---
## 4. 중요도와 피처 선택

> **이 장의 흐름**
>
> - 개발 데이터를 fit 70 · selection 36 · untouched evaluation 36으로 나눠 역할 분리
> - selection에서 피처를 고르고 untouched evaluation에서 한 번 비교 → 상관 · RF 내부 GridSearch

<img src="images/pfm7_perm.png" alt="selection 36개에서의 permutation importance" width="620">

**파이프라인 위치** · ③ 피처 선택 — 역할 분리

### 4-1. `B` 역할을 나눈 permutation importance · `permutation_importance` · `n_repeats=20` <sub>원본 셀 45 ~ 49 (선택 실험 A)</sub>

> 🎯 **selection 36개에서 피처를 하나씩 섞으면 정확도는 얼마나 떨어지는가?**

💡 **쉽게 말하면**

- 피처 하나를 **섞어서** 망가뜨렸을 때 점수가 얼마나 떨어지나 = 그 피처에 기댄 정도.
- fit · selection · untouched evaluation 세 역할을 나눔 (FINAL_HOLDOUT은 안 씀).

**실행 흐름**

1. 개발 142 → fit 70 · selection 36 · evaluation 36
2. fit으로 RF 학습
3. selection에서 20번 섞어 평균 감소

**먼저 예측** · selection 정확도가 이미 36/36이면 importance는 어떻게 보일까? — 답은 코드 실행 결과 아래에

In [10]:
from sklearn.inspection import permutation_importance  # 섞어서 성능 변화 보기

X_work, X_feat_eval, y_work, y_feat_eval = train_test_split(        # 25% untouched evaluation
    Xd, yd, test_size=0.25, stratify=yd, random_state=42)
X_feat_fit, X_feat_sel, y_feat_fit, y_feat_sel = train_test_split(  # 나머지를 fit · selection
    X_work, y_work, test_size=1 / 3, stratify=y_work, random_state=43)
sizes = [len(y_feat_fit), len(y_feat_sel), len(y_feat_eval)]        # 세 역할 크기
print("fit / selection / untouched evaluation =", " / ".join(map(str, sizes)))  # 70 / 36 / 36
RF300 = dict(n_estimators=300, max_features="sqrt", n_jobs=-1, random_state=42)  # 이 장의 RF 설정
rf_fit = RandomForestClassifier(**RF300)                            # 중요도 계산용
rf_fit.fit(X_feat_fit, y_feat_fit)                                  # fit 70개로만
base_ok = int((rf_fit.predict(X_feat_sel) == y_feat_sel).sum())     # selection 정답 수
print(f"Selection baseline: {base_ok}/{len(y_feat_sel)} = {base_ok / len(y_feat_sel):.4f}")  # 천장
# selection에서만
feature_perm = permutation_importance(rf_fit, X_feat_sel, y_feat_sel, n_repeats=20,
                                      scoring="accuracy", n_jobs=-1, random_state=42)
perm = pd.Series(feature_perm.importances_mean, index=wine.feature_names)  # 피처별 평균 감소
perm = perm.sort_values(ascending=False)                            # 큰 순서
print(perm.head(6).round(6).to_string())                            # 위 6개
print("0인 피처 수:", int((perm == 0).sum()), "/ 13")               # 감소가 관찰되지 않은 피처

fit / selection / untouched evaluation = 70 / 36 / 36


Selection baseline: 36/36 = 1.0000


color_intensity    0.043056
proline            0.027778
alcohol            0.026389
hue                0.019444
flavanoids         0.009722
magnesium          0.000000
0인 피처 수: 8 / 13


**결과** · 예측 정답은 대부분 0 — 천장(36/36)이라 떨어질 여지가 작음

- `fit / selection / untouched evaluation = 70 / 36 / 36` — 개발 데이터를 세 역할로
- `Selection baseline: 36/36 = 1.0000` — 이미 만점 → 1개 = 1/36 = 0.0278씩만 움직임
- `color_intensity` — 섞으면 가장 많이 떨어진 피처
- `0인 피처 수: 8 / 13` — 8개는 감소가 관찰되지 않음 ≠ 중요하지 않음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">fit</td>
<td style="text-align: left;">중요도 계산용 모델 학습</td>
<td style="text-align: left;">70개</td>
</tr>
<tr>
<td style="text-align: left;">selection</td>
<td style="text-align: left;">permutation importance · 고를 피처 결정</td>
<td style="text-align: left;">36개</td>
</tr>
<tr>
<td style="text-align: left;">untouched evaluation</td>
<td style="text-align: left;">전체 vs 선택 피처 모델을 한 번 비교</td>
<td style="text-align: left;">36개 (4-2)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 0.0000 = 이 모델 · 이 작은 selection · 이 점수에서 감소가 관찰되지 않았다는 뜻 · 피처가 쓸모없다는 판정이 아님
> - 작은 n · 천장 1.0000 · 정확도의 거친 해상도(1/36)가 함께 작용

> ✅ **한 줄 정리** · permutation importance는 이 모델 · 이 데이터 · 이 점수에서 피처를 섞었을 때의 성능 변화일 뿐이라, 0을 중요하지 않다는 판정으로 읽지 않는다.

**딥러닝 코드에서는**

- 입력 채널 · 토큰을 가려 성능 변화를 보는 occlusion · ablation 실험

### 4-2. `B` 고른 피처 vs 전체 피처 · `important_mask` · `untouched evaluation` <sub>원본 셀 50 · 52</sub>

> 🎯 **selection에서 고른 피처만 쓰면 untouched evaluation에서도 전체 피처와 같거나 더 좋은가?**

💡 **쉽게 말하면**

- selection 점수로 고른 뒤 **selection 점수로 성공을 주장하지 않음**.
- 선택에 안 쓴 evaluation에서 한 번 비교 · 반례를 숨기지 않음.

**실행 흐름**

1. importance > 0인 피처 mask
2. fit + selection으로 두 모델 재학습
3. evaluation 36개에서 한 번 비교

**먼저 예측** · 선택 피처 모델이 이길까? — 답은 코드 실행 결과 아래에

In [11]:
important_mask = feature_perm.importances_mean > 0.0  # selection에서 고정한 mask
print("고른 피처:", np.array(wine.feature_names)[important_mask].tolist())         # 5개
X_refit = np.concatenate([X_feat_fit, X_feat_sel])                                 # fit + selection
y_refit = np.concatenate([y_feat_fit, y_feat_sel])                                 # 같은 학습 풀
rf_all, rf_sel = RandomForestClassifier(**RF300), RandomForestClassifier(**RF300)  # 같은 설정 두 개
ok_all = int((rf_all.fit(X_refit, y_refit).predict(X_feat_eval) == y_feat_eval).sum())  # 전체 피처
rf_sel.fit(X_refit[:, important_mask], y_refit)                                    # 고른 피처만
ok_sel = int((rf_sel.predict(X_feat_eval[:, important_mask]) == y_feat_eval).sum())  # 고른 피처
print(f"All features: {ok_all}/36 = {ok_all / 36:.4f} · "                          # 전체 피처
      f"Selected features: {ok_sel}/36 = {ok_sel / 36:.4f}")
print(f"Observed difference (selected - all): {(ok_sel - ok_all) / 36:+.4f}")      # 1개 = 0.0278

고른 피처: ['alcohol', 'flavanoids', 'color_intensity', 'hue', 'proline']


All features: 36/36 = 1.0000 · Selected features: 35/36 = 0.9722
Observed difference (selected - all): -0.0278


**결과** · 예측 정답은 아님 — 전체 1.0000 > 선택 0.9722

- `고른 피처: ['alcohol', 'flavanoids', 'color_intensity', 'hue', 'proline']` — 13개 중 5개
- `All features: 36/36 = 1.0000 · Selected features: 35/36 = 0.9722` — 선택 피처가 1개 더 틀림
- `Observed difference (selected - all): -0.0278` — 1/36 차이 · 반례를 그대로 기록

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">selection 점수</td>
<td style="text-align: left;">피처를 고를 때 씀</td>
<td style="text-align: left;">성공 근거로 쓰지 않기</td>
</tr>
<tr>
<td style="text-align: left;">untouched evaluation</td>
<td style="text-align: left;">선택에 안 쓴 데이터</td>
<td style="text-align: left;">한 번 비교</td>
</tr>
<tr>
<td style="text-align: left;">반복 split · nested CV</td>
<td style="text-align: left;">더 안정적인 결론</td>
<td style="text-align: left;">피처 선택을 진지하게 할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 성공 기준은 선택 피처가 이기는 것이 아니라 **반례를 숨기지 않는 것**
> - 이 작은 단일 split의 관찰값 · final 성능이나 보편적 성공 주장이 아님

> ✅ **한 줄 정리** · selection에서 고른 피처는 선택에 쓰지 않은 데이터로 한 번만 비교하며, 이번에는 전체 피처(36/36)가 선택 피처(35/36)보다 나았다.

**딥러닝 코드에서는**

- 피처 · 토큰을 줄인 모델도 별도 val로 확인해야 함

### 4-3. `B` 상관된 피처 · RF 내부 GridSearch · `corr()` · `GridSearchCV` <sub>원본 셀 51 · 53 ~ 56 (선택 실험 B)</sub>

> 🎯 **서로 강하게 상관된 피처는 어떤 쌍이고, RF 하이퍼파라미터 36조합의 1위는 하나인가?**

💡 **쉽게 말하면**

- 상관된 피처는 서로를 대신해 **개별 중요도가 작게** 보일 수 있음 (가능한 현상).
- RF 내부 탐색은 family 비교와 다른 질문.

**실행 흐름**

1. fit 70개로 상관 행렬 · |r| > 0.7 쌍
2. RF 후보 3 × 4 × 3 = 36조합 GridSearch
3. 최고 점수 동점 수

**먼저 예측** · 36조합 중 최고 점수는 몇 개가 동점일까? — 답은 코드 실행 결과 아래에

In [12]:
from sklearn.model_selection import GridSearchCV  # 하이퍼파라미터 탐색

corr = pd.DataFrame(X_feat_fit, columns=wine.feature_names).corr()           # fit 부분만 기술
names = wine.feature_names                                                   # 짧은 이름
for i in range(13):                                                          # 모든 쌍
    for j in range(i + 1, 13):                                               # 중복 없이
        if abs(corr.iloc[i, j]) > 0.7:                                       # 강한 상관
            print(f"  {names[i]} ↔ {names[j]}  r = {corr.iloc[i, j]:+.3f}")  # 원본 셀 51
grid = {"n_estimators": [100, 200, 400], "max_depth": [None, 10, 20, 30],    # 3 × 4 × 3 = 36
        "max_features": ["sqrt", "log2", 0.5]}
gs_rf = GridSearchCV(RandomForestClassifier(random_state=42), grid,  # 원본 셀 56 · 조합 단위로 병렬
                     scoring="accuracy", cv=cv, n_jobs=-1).fit(Xd, yd)
top = pd.DataFrame(gs_rf.cv_results_)["mean_test_score"]                     # 36조합 점수
print("best_params_:", gs_rf.best_params_, f"· best_score_ {gs_rf.best_score_:.4f}")  # 순서상 먼저
print("최고 점수 동점 조합 수:", int((top == top.max()).sum()), "/ 36")      # 유일한 정답이 아님

  total_phenols ↔ flavanoids  r = +0.884
  total_phenols ↔ od280/od315_of_diluted_wines  r = +0.748
  flavanoids ↔ od280/od315_of_diluted_wines  r = +0.797


best_params_: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 100} · best_score_ 0.9791
최고 점수 동점 조합 수: 28 / 36


**결과** · 예측 정답은 28개 — best_params_는 동점 중 하나

- `total_phenols ↔ flavanoids  r = +0.884` — 가장 강한 상관
- `flavanoids ↔ od280/od315_of_diluted_wines  r = +0.797` — flavanoids와 od280도 비슷한 정보
- `best_params_: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 100} · best_score_ 0.9791` — 순서상 먼저인 동점 후보
- `최고 점수 동점 조합 수: 28 / 36` — 36조합 중 28개가 같은 점수 → 이 범위에서는 설정에 둔감

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">상관된 피처</td>
<td style="text-align: left;">서로를 대신 → 개별 importance가 작게 보일 수 있음</td>
<td style="text-align: left;">여러 개를 한꺼번에 지우지 않기</td>
</tr>
<tr>
<td style="text-align: left;">RF 내부 GridSearch</td>
<td style="text-align: left;">RF family 안의 설정 비교</td>
<td style="text-align: left;">family 승자를 정하는 실험이 아님</td>
</tr>
<tr>
<td style="text-align: left;">family 비교 (5장)</td>
<td style="text-align: left;">RF vs ExtraTrees를 같은 CV로</td>
<td style="text-align: left;">공식 선택</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 상관은 **가능한 메커니즘** · 4-1의 0.0000 원인으로 확정하지 않음 (작은 n · 천장 · 거친 해상도도 함께)

> ✅ **한 줄 정리** · 상관된 피처는 서로 중요도를 나눠 가질 수 있고, RF 설정 탐색의 1위는 동점 중 하나라서 결과는 계약과 함께 읽는다.

**딥러닝 코드에서는**

- 비슷한 입력 채널이 여럿이면 기여도 해석이 흩어짐

---
## 5. RF vs ExtraTrees · 최종 1회

> **이 장의 흐름**
>
> - RF · ExtraTrees를 같은 개발 CV로 비교 → CV mean 1위 family 고정
> - 선택 종료(FROZEN) 뒤 FINAL_HOLDOUT을 정확히 한 번 열어 보고

<img src="images/pfm7_family.png" alt="RF vs ExtraTrees 개발 CV · 고정 → FINAL_HOLDOUT 1회" width="763">

**파이프라인 위치** · ⑤ 선택 종료 → ⑥ FINAL_HOLDOUT 1회

### 5-1. `A` family 비교 · `ExtraTreesClassifier` · `bootstrap=False` <sub>원본 셀 57 · 58</sub>

> 🎯 **분할 임계값까지 무작위로 하는 ExtraTrees는 RF보다 개발 CV가 높은가?**

💡 **쉽게 말하면**

- **ExtraTrees** = 피처 후보 + **기준값까지 무작위** · 기본은 bootstrap 없이 전체 표본.
- 분산이 줄 수도, 편향이 커질 수도 → 결과로 확인.

**실행 흐름**

1. RF · ET 300그루
2. 같은 CV 평균 · 표준편차
3. CV mean 1위 family

**먼저 예측** · 어느 family가 이번 실행의 1위일까? — 답은 코드 실행 결과 아래에

In [13]:
rf_candidate = RandomForestClassifier(**RF300)                                # 4장과 같은 RF 설정
# 기준값까지 무작위
et_candidate = ExtraTreesClassifier(n_estimators=300, max_features="sqrt", bootstrap=False,
                                    n_jobs=-1, random_state=42)
family_models = {"Random Forest": rf_candidate, "ExtraTrees": et_candidate}   # 공식 후보 둘
rows = []                                                                     # family별 점수
for name, model in family_models.items():                                     # 같은 CV 계약
    s = cross_val_score(model, Xd, yd, cv=cv, scoring="accuracy", n_jobs=-1)  # 정확도 5개
    rows.append([name, s.mean(), s.std()])                                    # 한 줄
extra_evidence = pd.DataFrame(rows, columns=["Model", "CV mean", "CV std"])   # 표
extra_evidence = extra_evidence.sort_values("CV mean", ascending=False)       # 1등부터
print(extra_evidence.round(3).to_string(index=False))  # 원본은 Styler로만 출력

        Model  CV mean  CV std
   ExtraTrees    0.986   0.017
Random Forest    0.979   0.028


**결과** · 예측 정답은 ExtraTrees (0.986)

- `ExtraTrees    0.986   0.017` — 이번 실행의 1위 · 흔들림도 작음
- `Random Forest    0.979   0.028` — 차이 0.007 < 표준편차

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Random Forest</td>
<td style="text-align: left;">bootstrap + 피처 후보 무작위</td>
<td style="text-align: left;">기본 출발점</td>
</tr>
<tr>
<td style="text-align: left;">ExtraTrees</td>
<td style="text-align: left;">피처 후보 + 기준값 무작위 · bootstrap 없음</td>
<td style="text-align: left;">더 빠르고 더 무작위</td>
</tr>
<tr>
<td style="text-align: left;">평균 차이</td>
<td style="text-align: left;">통계적 유의성 주장이 아님</td>
<td style="text-align: left;">이번 실행의 1위만 고정</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 어느 family가 이긴다는 보편 규칙은 없음 (6장 시나리오 실험)
> - RF 내부 GridSearch(4-3)와 family 비교는 서로 다른 질문

> ✅ **한 줄 정리** · RF와 ExtraTrees는 같은 개발 CV로 비교해 이번 실행의 CV mean 1위를 고정하며, 작은 차이를 보편 법칙으로 읽지 않는다.

**딥러닝 코드에서는**

- 구조(family) 선택도 val로 · 작은 차이는 seed를 바꾸면 뒤집힐 수 있음

### 5-2. `A` 선택 고정 · FINAL_HOLDOUT 1회 · `FINAL_MODEL_FROZEN` · `clone` · `classification_report` <sub>원본 셀 59 ~ 62</sub>

> 🎯 **holdout 결과가 기대보다 낮으면 다른 family로 다시 바꿀 수 있는가?**

💡 **쉽게 말하면**

- 선택은 여기서 끝 · holdout 결과는 **재선택 근거가 아님**.
- 36개는 이번 공식 평가로 **소비(CONSUMED)** → 다시 새 holdout으로 쓰지 않음.

**실행 흐름**

1. CV mean 1위 family 고정
2. FROZEN → holdout 한 번 열기
3. 개발 142로 학습 → 36개 평가

**먼저 예측** · 36개 중 몇 개를 맞힐까? — 답은 코드 실행 결과 아래에

In [14]:
from sklearn.base import clone                     # 설정만 복사한 새 모델
from sklearn.metrics import classification_report  # 클래스별 표

family_scores = extra_evidence.set_index("Model")["CV mean"]      # 5-1 표
selected = family_scores.idxmax()                                 # 개발 CV 1위
sel_cv = family_scores[selected]                                  # 선택 근거 점수
print(f"선택된 family: {selected} · 근거 DEVELOPMENT_POOL CV mean {sel_cv:.3f}")  # 고정
FINAL_MODEL_FROZEN = True                                         # 선택 종료
X_final_once, y_final_once = open_final_holdout_once()            # 정확히 한 번
final_model = clone(family_models[selected]).fit(Xd, yd)          # 개발 142개로 학습
y_pred_final = final_model.predict(X_final_once)                  # 36개 예측
print("FINAL_HOLDOUT access count:", FINAL_HOLDOUT_ACCESS_COUNT)  # 1
print("FINAL_HOLDOUT Accuracy:", round(accuracy_score(y_final_once, y_pred_final), 3))  # 보고
tn_ = wine.target_names                                           # 품종 이름
report = classification_report(y_final_once, y_pred_final, target_names=tn_, digits=3)  # 표
print(report)                                                     # 클래스별 정밀도 · 재현율

선택된 family: ExtraTrees · 근거 DEVELOPMENT_POOL CV mean 0.986


FINAL_HOLDOUT access count: 1
FINAL_HOLDOUT Accuracy: 1.0
              precision    recall  f1-score   support

     class_0      1.000     1.000     1.000        12
     class_1      1.000     1.000     1.000        14
     class_2      1.000     1.000     1.000        10

    accuracy                          1.000        36
   macro avg      1.000     1.000     1.000        36
weighted avg      1.000     1.000     1.000        36



**결과** · 예측 정답은 36/36

- `선택된 family: ExtraTrees · 근거 DEVELOPMENT_POOL CV mean 0.986` — 개발 근거로만 고정
- `FINAL_HOLDOUT access count: 1` — 정확히 한 번
- `FINAL_HOLDOUT Accuracy: 1.0` — 36개 전부 정답
- `accuracy                          1.000        36` — 세 품종 모두 정밀도 · 재현율 1.000

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">개발 CV</td>
<td style="text-align: left;">선택 근거</td>
<td style="text-align: left;">여러 번</td>
</tr>
<tr>
<td style="text-align: left;">FINAL_HOLDOUT</td>
<td style="text-align: left;">선택된 하나를 한 번 평가</td>
<td style="text-align: left;">보고</td>
</tr>
<tr>
<td style="text-align: left;">holdout을 보고 재선택</td>
<td style="text-align: left;">holdout이 검증이 됨</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 1.0이 나왔다고 '완벽한 모델'이 아님 · 36개에서 1개 = 0.028
> - 낮게 나왔어도 재선택하면 안 됨 · 다시 고르려면 새 데이터

> ✅ **한 줄 정리** · 개발 근거로 family를 고정한 뒤 FINAL_HOLDOUT을 정확히 한 번 열어 보고하며, 그 결과로 다시 고르지 않는다.

**딥러닝 코드에서는**

- best.pt를 정한 뒤 test 1회 · 결과가 나빠도 그 test로 다시 튜닝하지 않음

---
## 6. 심화 · 이론은 경향

> **이 장의 흐름**
>
> - bootstrap=False면 OOB가 없는 이유 · Boosting과 OOB
> - 그다음 Clean · Noisy · 4가지 시나리오 · 피처 200개 중 5개에서 RF vs ET · max_features 실험

<img src="images/pfm7_scenarios.png" alt="4가지 시나리오 · RF vs ExtraTrees CV" width="632">

**파이프라인 위치** · ⑦ 확장 실험 (공식 선택은 바꾸지 않음)

### 6-1. `B` bootstrap=False와 OOB · `oob_score` · `bootstrap=False` <sub>원본 셀 63 ~ 68</sub>

> 🎯 **bootstrap=False로 OOB를 요청하면? 그때 트리들은 모두 같아지는가?**

💡 **쉽게 말하면**

- 전체 표본을 그대로 쓰면 **빠지는 샘플이 없음** → OOB 불가.
- 그래도 `max_features` 무작위가 남아 트리가 똑같아지지는 않음.

**실행 흐름**

1. bootstrap=False + oob_score=True 시도
2. bootstrap=False RF 두 그루의 첫 질문 비교

**먼저 예측** · bootstrap=False면 트리 300그루가 모두 같을까? — 답은 코드 실행 결과 아래에

In [15]:
try:  # 빠지는 샘플이 없는데 OOB를 요청
    # 오류 예상
    RandomForestClassifier(n_estimators=10, bootstrap=False, oob_score=True).fit(Xd, yd)
except ValueError as e:                                                # sklearn이 막음
    print("OOB 요청 결과:", str(e)[:58])                               # 앞부분만
# 표본 무작위 없음
rf_nb = RandomForestClassifier(n_estimators=300, bootstrap=False, max_features="sqrt",
                               n_jobs=-1, random_state=42).fit(Xd, yd)
roots = [est.tree_.feature[0] for est in rf_nb.estimators_]            # 그루마다 첫 질문
print("bootstrap=False 첫 질문 피처 종류:", len(set(roots)), "개")     # 피처 무작위는 남음
print("ExtraTrees 기본 bootstrap:", ExtraTreesClassifier().bootstrap)  # False

OOB 요청 결과: Out of bag estimation only available if bootstrap=True


bootstrap=False 첫 질문 피처 종류: 11 개
ExtraTrees 기본 bootstrap: False


**결과** · 예측 정답은 아님 — 첫 질문도 여러 피처로 흩어짐

- `OOB 요청 결과: Out of bag estimation only available if bootstrap=True` — 빠진 샘플이 없으면 OOB 불가
- `bootstrap=False 첫 질문 피처 종류:` — max_features 무작위가 남아 트리가 다양
- `ExtraTrees 기본 bootstrap: False` — ET는 기본으로 전체 표본 + 기준값 무작위

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">bootstrap=True</td>
<td style="text-align: left;">표본 무작위 · OOB 가능</td>
<td style="text-align: left;">RF 기본</td>
</tr>
<tr>
<td style="text-align: left;">bootstrap=False</td>
<td style="text-align: left;">표본 무작위 ↓ · OOB 불가 · 피처 무작위는 남음</td>
<td style="text-align: left;">ET 기본</td>
</tr>
<tr>
<td style="text-align: left;">Boosting (subsample=1.0)</td>
<td style="text-align: left;">빠진 표본 없음 → Bagging식 OOB 없음</td>
<td style="text-align: left;">subsample &lt; 1이면 OOB 추정 가능</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `bootstrap=False → 같은 트리 → 높은 분산` 같은 단정은 하지 않음
> - 'Boosting에는 구조적으로 OOB가 불가능'도 틀림 → subsample < 1.0인 stochastic GBM은 OOB 손실 추정 제공

> ✅ **한 줄 정리** · bootstrap=False면 빠지는 샘플이 없어 OOB를 쓸 수 없지만, 피처 무작위가 남아 트리가 똑같아지지는 않는다.

**딥러닝 코드에서는**

- val 없이 학습하면 중간 점검이 없음 · 내부 추정이 가능한 구조인지 먼저 확인

### 6-2. `B` Clean vs Noisy · RF vs ET · `practical_rf_et_label` <sub>원본 셀 69 ~ 73</sub>

> 🎯 **경계가 뚜렷한 데이터에서는 RF, 노이즈가 많은 데이터에서는 ET가 이긴다는 예상은 맞는가?**

💡 **쉽게 말하면**

- 예상(Prediction)을 먼저 적고 CV로 확인.
- 차이가 0.005 미만이면 '비슷' (노트북의 실용 표시 규칙 · 유의성 검정 아님).

**실행 흐름**

1. Clean 5000 × 5 · Noisy 5000 × 50
2. RF · ET 200그루 CV
3. 실용 표시 규칙

**먼저 예측** · Clean은 RF, Noisy는 ET가 이길까? — 답은 코드 실행 결과 아래에

In [16]:
def practical_rf_et_label(rf_mean, et_mean, threshold=0.005):  # 표시용 규칙 (유의성 검정 아님)
    diff = abs(rf_mean - et_mean)                              # 평균 차이
    if diff < threshold:                                       # 아주 작으면
        return "비슷 (명확한 승자 주장 어려움)", diff          # 판정 보류
    return ("RF" if rf_mean > et_mean else "ET"), diff         # 높은 쪽


clean = make_classification(n_samples=5000, n_features=5, n_informative=5,   # 경계 뚜렷 · 노이즈 0
                            n_redundant=0, class_sep=3.0, flip_y=0.0, random_state=42)
noisy = make_classification(n_samples=5000, n_features=50, n_informative=5,  # 노이즈 많음
                            n_redundant=10, class_sep=0.3, flip_y=0.3, random_state=42)
data = {"Clean": clean, "Noisy": noisy}                                      # 두 실험
for name, (Xs, ys) in data.items():                                          # 두 실험
    # RF
    rf = cross_val_score(RandomForestClassifier(200, random_state=42), Xs, ys, cv=cv, n_jobs=-1)
    et = cross_val_score(ExtraTreesClassifier(200, random_state=42), Xs, ys, cv=cv, n_jobs=-1)  # ET
    label, diff = practical_rf_et_label(rf.mean(), et.mean())                # 표시
    # 평균 ± 흔들림
    rf_t, et_t = f"{rf.mean():.4f} ± {rf.std():.4f}", f"{et.mean():.4f} ± {et.std():.4f}"
    print(f"{name}: RF {rf_t} · ET {et_t} · 차이 {diff:.4f} → {label}")      # 한 줄

Clean: RF 0.9990 ± 0.0009 · ET 0.9996 ± 0.0005 · 차이 0.0006 → 비슷 (명확한 승자 주장 어려움)


Noisy: RF 0.7116 ± 0.0119 · ET 0.7122 ± 0.0172 · 차이 0.0006 → 비슷 (명확한 승자 주장 어려움)


**결과** · 예측 정답은 판정 불가 — 둘 다 '비슷'

- `Clean: RF 0.9990 ± 0.0009 · ET 0.9996 ± 0.0005 · 차이 0.0006 → 비슷 (명확한 승자 주장 어려움)` — 예상(RF 우세)과 다름
- `Noisy: RF 0.7116 ± 0.0119 · ET 0.7122 ± 0.0172 · 차이 0.0006 → 비슷 (명확한 승자 주장 어려움)` — 예상(ET 우세)과 다름 · 원본(sklearn 1.6)은 RF 0.7150 · ET 0.7104

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Clean 예상</td>
<td style="text-align: left;">RF 우세</td>
<td style="text-align: left;">결과: 차이 0.0006 → 비슷</td>
</tr>
<tr>
<td style="text-align: left;">Noisy 예상</td>
<td style="text-align: left;">ET 우세</td>
<td style="text-align: left;">결과: 차이 0.0006 → 비슷</td>
</tr>
<tr>
<td style="text-align: left;">더 강한 비교</td>
<td style="text-align: left;">반복 split · fold 수 · 표본 수</td>
<td style="text-align: left;">Better Question</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 두 예상은 '틀림'으로 끝난 것이 아니라 **이 규칙 · 이 해상도로는 명확한 승자를 주장하기 어려움**
> - 승자를 만들려고 seed를 바꾸거나 threshold를 낮추지 않음

> ✅ **한 줄 정리** · Clean에서는 RF, Noisy에서는 ET라는 예상은 이번 실험에서 둘 다 '비슷'으로 판정 보류됐고, 그 결과를 그대로 남긴다.

**딥러닝 코드에서는**

- '이 구조가 노이즈에 강하다'는 이론도 실험으로 확인 · 작은 차이는 보류

### 6-3. `B` 상황별 시나리오 4개 · `scenarios` <sub>원본 셀 74 · 75</sub>

> 🎯 **데이터 상황 4가지에서 RF와 ET의 승자는 한쪽으로 정해지는가?**

💡 **쉽게 말하면**

- 상황마다 예상 승자를 먼저 적고 실행.
- 실제 승자가 다르면 결과를 남기고 다음 질문으로.

**실행 흐름**

1. 시나리오 4개 · 2000개씩
2. RF · ET 100그루 CV
3. 실용 표시

**먼저 예측** · 4개 시나리오의 승자는 모두 같을까? — 답은 코드 실행 결과 아래에

In [17]:
scenarios = {"경계 뚜렷 + feature 적음": (10, 8, 0.0, 2.0),    # (특성 수, 유용, 라벨 노이즈, 분리)
             "노이즈 많음 + feature 많음": (50, 5, 0.3, 0.3),
             "노이즈 많음 + feature 적음": (10, 5, 0.3, 0.5),
             "경계 뚜렷 + feature 많음(노이즈)": (50, 5, 0.0, 2.0)}
# 원본 셀 75 (특성 수 · 유용 · 라벨 노이즈 · 분리)
for name, (nf, ni, flip, sep) in scenarios.items():
    Xs, ys = make_classification(                              # 시나리오 데이터 2000개
        n_samples=2000, n_features=nf, n_informative=ni, n_redundant=0,
        flip_y=flip, class_sep=sep, random_state=42)
    # RF
    rf = cross_val_score(RandomForestClassifier(100, random_state=42), Xs, ys, cv=cv, n_jobs=-1)
    et_m = ExtraTreesClassifier(100, bootstrap=False, random_state=42)  # ET
    et = cross_val_score(et_m, Xs, ys, cv=cv, n_jobs=-1)       # ET
    label, diff = practical_rf_et_label(rf.mean(), et.mean())  # 6-2 규칙
    print(f"{name}: RF {rf.mean():.4f} · ET {et.mean():.4f} · 차이 {diff:.4f} → {label}")  # 한 줄

경계 뚜렷 + feature 적음: RF 0.9590 · ET 0.9695 · 차이 0.0105 → ET


노이즈 많음 + feature 많음: RF 0.6385 · ET 0.6105 · 차이 0.0280 → RF


노이즈 많음 + feature 적음: RF 0.7330 · ET 0.7270 · 차이 0.0060 → RF


경계 뚜렷 + feature 많음(노이즈): RF 0.9735 · ET 0.9780 · 차이 0.0045 → 비슷 (명확한 승자 주장 어려움)


**결과** · 예측 정답은 아님 — ET · RF · RF · 비슷

- `경계 뚜렷 + feature 적음: RF 0.9590 · ET 0.9695 · 차이 0.0105 → ET` — 뚜렷한 경계에서 오히려 ET
- `노이즈 많음 + feature 많음: RF 0.6385 · ET 0.6105 · 차이 0.0280 → RF` — 노이즈에서 오히려 RF
- `노이즈 많음 + feature 적음: RF 0.7330 · ET 0.7270 · 차이 0.0060 → RF` — RF
- `경계 뚜렷 + feature 많음(노이즈): RF 0.9735 · ET 0.9780 · 차이 0.0045 → 비슷 (명확한 승자 주장 어려움)` — 보류 · 원본 값과 끝자리만 다름 (sklearn 버전)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">이론의 예상</td>
<td style="text-align: left;">Clean → RF · Noisy → ET</td>
<td style="text-align: left;">경향일 뿐</td>
</tr>
<tr>
<td style="text-align: left;">이번 결과</td>
<td style="text-align: left;">Clean 쪽 ET · Noisy 쪽 RF</td>
<td style="text-align: left;">예상과 반대인 시나리오도 있음</td>
</tr>
<tr>
<td style="text-align: left;">다음 질문</td>
<td style="text-align: left;">데이터 크기 · 분리 난도 · fold 변동성</td>
<td style="text-align: left;">반례를 질문으로 바꿈</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 실제 승자가 예상과 달라도 그대로 남김 · 시나리오를 다시 설계해 승자를 만들지 않음

> ✅ **한 줄 정리** · 데이터 상황별 RF · ET 승자는 이론의 예상과 반대로 나온 경우도 있어, 이론은 경향이고 표가 근거다.

**딥러닝 코드에서는**

- '이 데이터엔 이 구조'라는 경험칙도 내 데이터에서 다시 확인

### 6-4. `B` 피처 200개 중 5개 · max_features · `max_features` · `chance reference` <sub>원본 셀 76 ~ 78</sub>

> 🎯 **피처 200개 중 중요한 것이 5개뿐일 때 max_features를 바꾸면 CV가 단조롭게 변하는가?**

💡 **쉽게 말하면**

- 후보를 줄이면 다른 피처를 볼 가능성 ↑ · 중요한 피처를 놓칠 가능성도 ↑.
- 그래서 '작을수록 분산이 단조롭게 준다'고 미리 결론 내지 않음.

**실행 흐름**

1. 300 × 200 · 중요 5개 · 약한 신호
2. max_features 5가지
3. 다수 클래스 비율(찍기 기준)과 비교

**먼저 예측** · 가장 좋은 설정은 찍기 기준선보다 확실히 높을까? — 답은 코드 실행 결과 아래에

In [18]:
X_many, y_many = make_classification(n_samples=300, n_features=200, n_informative=5,  # 원본 셀 77
                                     n_redundant=0, flip_y=0.1, class_sep=0.3, random_state=42)
# 후보 5개
opts = {"sqrt (14개)": "sqrt", "log2 (7개)": "log2", "5개": 5, "50개": 50, "None (200개)": None}
rows = []                                                                # 설정별 점수
for name, mf in opts.items():                                            # 설정마다
    rf = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=42)  # 같은 RF
    s = cross_val_score(rf, X_many, y_many, cv=cv, scoring="accuracy", n_jobs=-1)  # 5-Fold 병렬
    rows.append([name, s.mean(), s.std()])                               # 한 줄
res = pd.DataFrame(rows, columns=["max_features", "CV mean", "CV std"])  # 표
print(res.round(4).to_string(index=False))                               # 원본 표
chance = np.bincount(y_many).max() / len(y_many)                         # 다수 클래스 비율
best = res.loc[res["CV mean"].idxmax()]                                  # 최고 행
n_below = int((res["CV mean"] < chance).sum())                           # chance 미만 설정 수
gap = best["CV mean"] - chance                                           # 최고와 chance 차이
print(f"chance {chance:.3f} · chance 미만 {n_below}/5개 · "              # 근거 판단
      f"최고와 chance 차이 {gap:+.4f} (std {best['CV std']:.4f})")

max_features  CV mean  CV std
  sqrt (14개)   0.5400  0.0389
   log2 (7개)   0.4900  0.0688
          5개   0.5200  0.1008
         50개   0.4867  0.1108
 None (200개)   0.4600  0.0712
chance 0.530 · chance 미만 4/5개 · 최고와 chance 차이 +0.0100 (std 0.0389)


**결과** · 예측 정답은 아님 — 신호가 chance 부근

- `sqrt (14개)   0.5400  0.0389` — 가장 높지만
- `None (200개)   0.4600  0.0712` — 전부 쓰면 가장 낮음
- `chance 0.530 · chance 미만 4/5개 · 최고와 chance 차이 +0.0100 (std 0.0389)` — 차이가 std 안 → 효과를 주장할 근거가 아님

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">max_features 작게</td>
<td style="text-align: left;">다양성 ↑ · 중요한 피처를 놓칠 수 있음</td>
<td style="text-align: left;">피처가 많고 상관될 때 후보</td>
</tr>
<tr>
<td style="text-align: left;">max_features 크게</td>
<td style="text-align: left;">다양성 ↓ · bias ↓</td>
<td style="text-align: left;">중요 피처가 적고 확실할 때</td>
</tr>
<tr>
<td style="text-align: left;">이번 결과</td>
<td style="text-align: left;">단조 패턴 아님 · chance 부근</td>
<td style="text-align: left;">주장하지 않음</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - n = 300 · 5 folds · 약한 신호의 가짜 이진 데이터 → 작은 차이를 보편 법칙으로 일반화하지 않음
> - 데이터 · seed · threshold를 바꿔 승자를 만들지 않으며, 이 결과로 공식 family를 재선택하지 않음

> ✅ **한 줄 정리** · 신호가 찍기 기준선 부근인 데이터에서는 max_features 효과를 주장할 수 없으며, 그 한계를 그대로 기록한다.

**딥러닝 코드에서는**

- 학습이 chance 근처면 하이퍼파라미터 비교 전에 데이터 · 신호부터 점검

---
## 7. 종합 실습 · 유방암 데이터로 같은 계약 한 번 더

> **이 장의 흐름**
>
> - 1 ~ 5장 절차를 한 번에 사용
> - Wine의 holdout은 이미 소비됐으므로 유방암 569개로 새 계약을 만들어 끝까지

<img src="images/pfm7_practice.png" alt="유방암 개발 CV · 후보 5개 → 1위 고정 → holdout 1회" width="644">

**파이프라인 위치** · ② 잠금 → ⑤ 교차검증 · 선택 → ⑥ holdout 1회

> 🎯 **트리 · Bagging · RF · ExtraTrees · Soft Voting 중 개발 CV 1위를 고정하면 holdout 점수는?**

💡 **쉽게 말하면**

- **계약은 그대로, 데이터만 새로**.
- holdout은 선택이 끝난 뒤 한 번 · 결과로 다시 고르지 않음.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">유방암 569 → 개발 455 · 잠금 114</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">후보 5개 정의</td>
<td style="text-align: left;">2-2 · 3-3 · 5-1</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">같은 CV 평균 · 표준편차 → 1위 고정</td>
<td style="text-align: left;">3-3 · 5-1</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">holdout 1회 · 보고</td>
<td style="text-align: left;">5-2</td>
</tr>
</tbody>
</table>

**먼저 예측** · 개발 CV 1위는 어느 모델일까? holdout은 CV보다 높을까 낮을까? — 답은 코드 실행 결과 아래에

In [19]:
# 1단계: 새 데이터로 새 계약 (Wine holdout은 이미 소비)
from sklearn.datasets import load_breast_cancer  # 유방암 569건 · 30특성

bc = load_breast_cancer()                                                 # 0 = 악성, 1 = 양성
Xbd, Xbh, ybd, ybh = train_test_split(bc.data, bc.target, test_size=0.2,  # 잠금 20%
                                      stratify=bc.target, random_state=42)
print("개발", Xbd.shape, "· 잠금", Xbh.shape)                             # 455 · 114


# 2 · 3단계: 후보 5개를 같은 CV로 → 1위 고정
lr_bc = Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=5000))])  # 투표 구성원
cands = {"Single Tree": DT(random_state=42),                                # 기준
         "Bagging": BaggingClassifier(DT(), n_estimators=200, n_jobs=-1, random_state=42),
         "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
         "ExtraTrees": ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42),
         "Soft Voting": VotingClassifier([("lr", lr_bc),                    # 서로 다른 셋
                                          ("rf", RandomForestClassifier(300, random_state=42)),
                                          ("dt", DT(max_depth=4, random_state=42))],
                                         voting="soft")}
scores = {}                                                                 # 후보별 CV 평균
for name, m in cands.items():                                               # 같은 계약
    s = cross_val_score(m, Xbd, ybd, cv=cv, scoring="accuracy", n_jobs=-1)  # 정확도 5개
    scores[name] = s.mean()                                                 # 저장
    print(f"{name:<14s} CV {s.mean():.4f} ± {s.std():.4f}")                 # 표
pick = max(scores, key=scores.get)                                          # 개발 CV 1위
print("고정:", pick)                                                        # 선택 종료

개발 (455, 30) · 잠금 (114, 30)
Single Tree    CV 0.9165 ± 0.0179


Bagging        CV 0.9516 ± 0.0215


Random Forest  CV 0.9648 ± 0.0176


ExtraTrees     CV 0.9736 ± 0.0164


Soft Voting    CV 0.9538 ± 0.0224
고정: ExtraTrees


In [20]:
# 4단계: holdout 1회 (고정한 후보 그대로)
final_bc = clone(cands[pick]).fit(Xbd, ybd)                       # 개발 455개로 학습
acc_h = accuracy_score(ybh, final_bc.predict(Xbh))                # 잠금 114개 · 처음이자 마지막
print(f"holdout 114개 정확도: {acc_h:.4f} · 개발 CV {scores[pick]:.4f}")  # 보고
print("틀린 건수:", int(round((1 - acc_h) * len(ybh))), "/ 114")  # 1건 = 0.0088

holdout 114개 정확도: 0.9561 · 개발 CV 0.9736
틀린 건수: 5 / 114


**결과** · 예측 정답은 ExtraTrees 1위 · holdout은 CV보다 조금 낮음

- `개발 (455, 30) · 잠금 (114, 30)` — 8 : 2 잠금
- `Single Tree    CV 0.9165 ± 0.0179` — 트리 한 그루가 가장 낮음
- `ExtraTrees     CV 0.9736 ± 0.0164` — 1위 · Wine과 같은 family
- `Soft Voting    CV 0.9538 ± 0.0224` — Voting은 RF · ET보다 낮음
- `고정: ExtraTrees` — 개발 근거로만 선택
- `holdout 114개 정확도: 0.9561 · 개발 CV 0.9736` — 보고할 점수 · CV보다 0.0175 낮음 · 재선택하지 않음
- `틀린 건수: 5 / 114` — 114개에서 1건 = 0.0088

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Wine (2 · 5장)</td>
<td style="text-align: left;">Voting이 단일 모델을 못 넘음 · ET 1위 · holdout 36/36</td>
<td style="text-align: left;">작은 holdout</td>
</tr>
<tr>
<td style="text-align: left;">유방암 (이 장)</td>
<td style="text-align: left;">ET 1위 · holdout 0.9561 (CV보다 낮음)</td>
<td style="text-align: left;">같은 계약 · 다른 숫자</td>
</tr>
<tr>
<td style="text-align: left;">공통</td>
<td style="text-align: left;">잠금 → 같은 CV → 고정 → 1회</td>
<td style="text-align: left;">바뀌지 않는 절차</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 두 데이터 모두 ET가 1위였다고 'ET가 항상 좋다'로 일반화하지 않음 (6-3에서 승자는 제각각)
> - holdout이 CV보다 높거나 낮아도 재선택하지 않음 · 114개에서 1건 = 0.0088

> ✅ **한 줄 정리** · 데이터가 바뀌면 승자는 바뀌어도, 잠금 → 같은 CV로 비교 → 고정 → holdout 1회라는 계약은 그대로다.

**딥러닝 코드에서는**

- 새 데이터셋마다 같은 실험 프로토콜(분할 · val 선택 · test 1회)을 다시 적용

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 흔들림</td>
<td style="text-align: left;">경계 불일치율 트리 0.108 → Bagging 0.061 · 균형점 복잡도 5</td>
<td style="text-align: left;">1-1 · 1-2</td>
</tr>
<tr>
<td style="text-align: left;">2. Voting 반례</td>
<td style="text-align: left;">Logistic 0.979 &gt; Hard 0.972 &gt; Soft 0.965 · disagreement ≠ 좋은 조합</td>
<td style="text-align: left;">2-2 · 2-3</td>
</tr>
<tr>
<td style="text-align: left;">3. Bagging → OOB → RF</td>
<td style="text-align: left;">CV 0.944 = OOB 0.944 · 빠진 비율 0.366 · RF 0.979 ± 0.028</td>
<td style="text-align: left;">3-1 ~ 3-3</td>
</tr>
<tr>
<td style="text-align: left;">4. 역할 분리</td>
<td style="text-align: left;">selection 36/36 천장 · 0인 피처 8개 · 선택 35/36 &lt; 전체 36/36</td>
<td style="text-align: left;">4-1 · 4-2</td>
</tr>
<tr>
<td style="text-align: left;">5. 선택 → 1회</td>
<td style="text-align: left;">ET 0.986로 고정 → holdout 36/36 · 이론은 경향 (시나리오 승자 제각각)</td>
<td style="text-align: left;">5-1 · 5-2 · 6장</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">bootstrap</td>
<td style="text-align: left;">복원추출 · 그루마다 약 37% 빠짐</td>
<td style="text-align: left;">데이터 증강</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">Bagging</td>
<td style="text-align: left;">같은 모델 · 다른 표본 · 평균 → 분산 ↓</td>
<td style="text-align: left;">모델 앙상블</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">Voting</td>
<td style="text-align: left;">서로 다른 모델 결합 · 항상 이기지 않음</td>
<td style="text-align: left;">체크포인트 앙상블</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">오류 겹침</td>
<td style="text-align: left;">다양성 + 구성원 품질 + 겹침</td>
<td style="text-align: left;">같은 구조끼리 효과 작음</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">OOB</td>
<td style="text-align: left;">빠진 샘플로 내부 추정 · holdout 아님</td>
<td style="text-align: left;">내부 추정</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">Random Forest</td>
<td style="text-align: left;">Bagging + max_features</td>
<td style="text-align: left;">dropout</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">평가 계약</td>
<td style="text-align: left;">잠금 → 같은 CV → 고정 → 1회</td>
<td style="text-align: left;">실험 프로토콜</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">ExtraTrees</td>
<td style="text-align: left;">기준값까지 무작위 · bootstrap 없음</td>
<td style="text-align: left;">더 강한 무작위</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">permutation importance</td>
<td style="text-align: left;">섞었을 때 성능 변화 · 0 ≠ 쓸모없음</td>
<td style="text-align: left;">occlusion · ablation</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">이론은 경향</td>
<td style="text-align: left;">시나리오별 승자는 실험으로</td>
<td style="text-align: left;">내 데이터에서 확인</td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">bootstrap만 바꾸면 트리와 Bagging의 경계 불일치율은?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">Bias와 Variance는?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">FINAL_HOLDOUT을 선택 전에 열면?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">Wine에서 Voting은 최고 단일 모델을 넘었나?</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">예측이 많이 다르면 좋은 조합인가?</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">OOB는 무엇이고 holdout인가?</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">2차원 데모에서 트리 vs Bagging CV는?</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">RF는 Bagging과 무엇이 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">permutation importance 0의 뜻은?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">고른 피처 vs 전체 피처 결과는?</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">상관된 피처의 importance는?</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">RF vs ET 중 이번 1위는?</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">holdout이 낮게 나오면 재선택해도 되나?</td>
</tr>
<tr>
<td style="text-align: left;">6-1</td>
<td style="text-align: left;">bootstrap=False면 OOB는?</td>
</tr>
<tr>
<td style="text-align: left;">6-2</td>
<td style="text-align: left;">Clean → RF, Noisy → ET 예상은?</td>
</tr>
<tr>
<td style="text-align: left;">6-3</td>
<td style="text-align: left;">시나리오별 승자는 이론대로였나?</td>
</tr>
<tr>
<td style="text-align: left;">6-4</td>
<td style="text-align: left;">피처 200개 중 5개 실험의 결론은?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. Wine 개발 데이터에서 `RandomForestClassifier(n_estimators=50, max_features="log2", random_state=42)`의 CV 평균을 소수 셋째 자리로 출력
2. 같은 모델을 `oob_score=True`로 개발 데이터 전체에 fit해 OOB 점수를 소수 셋째 자리로 출력

In [21]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 1-1에서 Bagging의 `n_estimators`를 80 → 10으로 줄이면 경계 불일치율은?
2. 2-2에서 트리 구성원을 빼고 로지스틱 · KNN만 Soft Voting하면 CV는?
3. 3-3에서 RF의 `max_features`를 None(13개 전부)으로 바꾸면 Bagging과 비슷해질까?

In [22]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">트리 0.108 · Bagging 0.061</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">중요한 패턴을 못 배움(과소) · 데이터 변화에 민감(과적합)</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">RuntimeError · 선택을 먼저 완료</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">아님 · Logistic 0.979 &gt; Hard 0.972 &gt; Soft 0.965</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">아님 · 구성원 품질 · 오류 겹침을 함께</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">빠진 샘플을 안 본 트리들의 표 · holdout 아님 · 보조 추정</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">0.944 ± 0.022 vs 0.952 ± 0.021</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">분기마다 피처 후보를 제한(max_features) → 트리들이 덜 닮음</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">이 조건에서 감소가 관찰되지 않음 · 중요하지 않다는 판정 아님</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">선택 35/36 &lt; 전체 36/36 · 반례를 숨기지 않음</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">서로 나눠 가져 작게 보일 수 있음 (가능한 메커니즘)</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">ExtraTrees 0.986 (차이는 유의성 주장 아님)</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">안 됨 · 다시 고르려면 새 데이터</td>
</tr>
<tr>
<td style="text-align: left;">6-1</td>
<td style="text-align: left;">불가 · 빠진 샘플이 없음</td>
</tr>
<tr>
<td style="text-align: left;">6-2</td>
<td style="text-align: left;">둘 다 '비슷'으로 판정 보류</td>
</tr>
<tr>
<td style="text-align: left;">6-3</td>
<td style="text-align: left;">아님 · ET · RF · RF · 비슷</td>
</tr>
<tr>
<td style="text-align: left;">6-4</td>
<td style="text-align: left;">chance 부근이라 max_features 효과 주장 불가</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
rf50 = RandomForestClassifier(n_estimators=50, max_features="log2", random_state=42)
print(round(cross_val_score(rf50, Xd, yd, cv=cv).mean(), 3))  # 0.972
rf50_oob = RandomForestClassifier(n_estimators=50, max_features="log2",
                                  oob_score=True, random_state=42)
print(round(rf50_oob.fit(Xd, yd).oob_score_, 3))  # 0.972
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">0.106 — 트리(0.108)와 거의 같아짐 · 그루가 적으면 평균 효과가 사라짐</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">0.972 — 약한 트리를 빼도 최고 단일(0.979)을 넘지 못함 · 두 구성원의 오류가 겹침(overlap 0.500)</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">0.944 — Bagging과 같아짐 · 피처 무작위가 없으면 RF ≈ Bagging</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) Voting은 항상 단일 모델보다 낫다고 답함</td>
<td style="text-align: left;">앙상블 = 무조건 개선으로 외움</td>
<td style="text-align: left;">구성원 품질 · 오류 겹침 · 같은 CV로 확인</td>
<td style="text-align: left;">최고 단일 모델 점수와 비교했나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. Bagging과 Random Forest는 어떻게 다른가요?**
> - Bagging은 같은 모델을 bootstrap 표본마다 학습해 예측을 평균내 분산을 줄입니다.
> - Random Forest는 여기에 분기마다 고려할 피처 후보를 제한하는 max_features를 더해 트리들이 덜 닮게 만듭니다.
> - Wine에서는 단일 트리 0.902, Bagging 0.944, RF 0.979로 올랐지만 이 데이터의 결과로만 해석합니다.

**Q2. OOB 점수는 무엇이고 test 점수 대신 써도 되나요?**
> - bootstrap에서 각 트리는 약 37%의 샘플을 보지 않는데, 그 샘플을 안 본 트리들의 표로 평가한 내부 추정치가 OOB입니다.
> - 개발 데이터 안의 보조 추정이라 최종 보고용 holdout을 대신하지 않고, fold별 평균과 표준편차를 주는 CV와도 역할이 다릅니다.

**Q3. 여러 모델을 합치면 항상 성능이 오르나요?**
> - 아닙니다.
> - Wine에서 로지스틱 0.979가 Hard Voting 0.972와 Soft Voting 0.965보다 높았습니다.
> - 앙상블 효과는 예측이 다른 정도뿐 아니라 구성원 품질과 오류가 겹치는 정도에 달려 있어서, 같은 CV 계약으로 최고 단일 모델과 비교해 확인합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>BaggingClassifier</code></td>
<td style="text-align: left;">sklearn.ensemble</td>
<td style="text-align: left;">같은 모델 · bootstrap 표본</td>
<td style="text-align: left;">1-1 <code>BaggingClassifier(estimator=DT(), n_estimators=80, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.random.default_rng().choice(replace=True)</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">복원추출</td>
<td style="text-align: left;">1-1 <code>choice(260, size=260, replace=True)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>VotingClassifier(voting=)</code></td>
<td style="text-align: left;">sklearn.ensemble</td>
<td style="text-align: left;">Hard · Soft Voting</td>
<td style="text-align: left;">2-2 <code>VotingClassifier(members, voting="soft")</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>cross_val_predict</code></td>
<td style="text-align: left;">sklearn.model_selection</td>
<td style="text-align: left;">fold 밖 예측</td>
<td style="text-align: left;">2-2 <code>cross_val_predict(model, Xd, yd, cv=cv)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>oob_score=True</code> · <code>oob_score_</code></td>
<td style="text-align: left;">Bagging · RF</td>
<td style="text-align: left;">OOB 추정</td>
<td style="text-align: left;">3-1 <code>bag_wine.oob_score_</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>estimators_samples_</code></td>
<td style="text-align: left;">Bagging 속성</td>
<td style="text-align: left;">그루마다 학습 샘플</td>
<td style="text-align: left;">3-1 <code>bag_wine.estimators_samples_</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>RandomForestClassifier(max_features=)</code></td>
<td style="text-align: left;">sklearn.ensemble</td>
<td style="text-align: left;">Bagging + 피처 무작위</td>
<td style="text-align: left;">3-3 <code>max_features="sqrt"</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>permutation_importance</code></td>
<td style="text-align: left;">sklearn.inspection</td>
<td style="text-align: left;">섞었을 때 성능 변화</td>
<td style="text-align: left;">4-1 <code>permutation_importance(rf_fit, X_feat_sel, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>GridSearchCV</code></td>
<td style="text-align: left;">sklearn.model_selection</td>
<td style="text-align: left;">RF 내부 설정 탐색</td>
<td style="text-align: left;">4-3 <code>GridSearchCV(RandomForestClassifier(…), grid, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>ExtraTreesClassifier</code></td>
<td style="text-align: left;">sklearn.ensemble</td>
<td style="text-align: left;">기준값까지 무작위</td>
<td style="text-align: left;">5-1 <code>ExtraTreesClassifier(n_estimators=300, bootstrap=False, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>clone</code></td>
<td style="text-align: left;">sklearn.base</td>
<td style="text-align: left;">설정만 복사한 새 모델</td>
<td style="text-align: left;">5-2 <code>clone(family_models[selected])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>classification_report</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">클래스별 표</td>
<td style="text-align: left;">5-2 <code>classification_report(y_final_once, …)</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">holdout을 먼저 잠금</td>
<td style="text-align: left;">선택 전에 holdout 보기</td>
</tr>
<tr>
<td style="text-align: left;">Voting은 최고 단일 모델과 비교</td>
<td style="text-align: left;">앙상블이면 무조건 낫다고 가정</td>
</tr>
<tr>
<td style="text-align: left;">OOB는 보조 추정으로</td>
<td style="text-align: left;">OOB를 holdout 대신 보고</td>
</tr>
<tr>
<td style="text-align: left;">피처 선택은 역할 분리</td>
<td style="text-align: left;">selection 점수로 성공 주장</td>
</tr>
<tr>
<td style="text-align: left;">예상과 다른 결과도 기록</td>
<td style="text-align: left;">seed · 데이터를 바꿔 승자 만들기</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Bagging vs RF</td>
<td style="text-align: left;">표본만 무작위 vs 표본 + 피처</td>
<td style="text-align: left;">RF는 트리들이 덜 닮음</td>
</tr>
<tr>
<td style="text-align: left;">RF vs ExtraTrees</td>
<td style="text-align: left;">bootstrap · 최적 기준값 vs 전체 표본 · 무작위 기준값</td>
<td style="text-align: left;">같은 CV로 고름</td>
</tr>
<tr>
<td style="text-align: left;">Hard vs Soft Voting</td>
<td style="text-align: left;">라벨 다수결 vs 확률 평균</td>
<td style="text-align: left;">확률 품질에 따라</td>
</tr>
<tr>
<td style="text-align: left;">OOB vs CV</td>
<td style="text-align: left;">빠진 샘플로 한 번 vs fold별 mean · std</td>
<td style="text-align: left;">선택은 CV</td>
</tr>
<tr>
<td style="text-align: left;">Bias vs Variance</td>
<td style="text-align: left;">덜 배움 vs 흔들림</td>
<td style="text-align: left;">Bagging은 분산 · Boosting은 편향</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개 · 2-3 disagreement 표를 말로 설명</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">7장 종합 실습을 다른 데이터(digits 등)로 처음부터</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 보강 3(Boosting) 시작 전 ✅ 줄만 읽기</td>
<td style="text-align: left;">15분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 노트북의 셀 79개를 Core 흐름대로 1 ~ 5장 · Optional Lab A · B는 4장 · Advanced Extension은 6장으로
- 원본에서 Styler 객체로만 출력되던 표(경계 불일치 · Voting · disagreement · 3모델 · RF vs ET)는 숫자를 직접 출력
- 원본 plotly 그림 9개 · `assets/` 슬라이드 12장(원본 폴더에 없음) · 첨부 이미지 1장은 장 그림과 💡 · 비교 표로
- 7장 종합 실습은 Wine holdout이 이미 소비됐으므로 유방암 데이터로 같은 계약을 다시 (새로 추가)

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">셀 0 · 1</td>
<td style="text-align: left;">오늘의 문제 · 목표 · 트리 복습</td>
<td style="text-align: left;">0. 원본 목표</td>
</tr>
<tr>
<td style="text-align: left;">셀 2 ~ 4</td>
<td style="text-align: left;">트리의 불안정 · 경계 비교</td>
<td style="text-align: left;">1-1 · 1장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 5 ~ 11</td>
<td style="text-align: left;">앙상블 지도 · Bias vs Variance</td>
<td style="text-align: left;">0. 힌트 · 1-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 16 ~ 18</td>
<td style="text-align: left;">Wine 잠금 · 기본 모델</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 12 ~ 21</td>
<td style="text-align: left;">Voting 질문 · Hard vs Soft · CV · PAUSE</td>
<td style="text-align: left;">2-2 · 2장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 22 ~ 25</td>
<td style="text-align: left;">disagreement · error overlap · Reveal</td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 26 ~ 30</td>
<td style="text-align: left;">Bagging → OOB</td>
<td style="text-align: left;">3-1 · 3장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 39 ~ 42</td>
<td style="text-align: left;">Optional Quick Demo</td>
<td style="text-align: left;">3-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 31 ~ 38 · 43 · 44</td>
<td style="text-align: left;">Random Forest · 하이퍼파라미터 · Core 비교</td>
<td style="text-align: left;">3-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 45 ~ 49</td>
<td style="text-align: left;">Optional Lab A · permutation</td>
<td style="text-align: left;">4-1 · 4장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 50 · 52</td>
<td style="text-align: left;">선택 피처 vs 전체</td>
<td style="text-align: left;">4-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 51 · 53 ~ 56</td>
<td style="text-align: left;">상관 · Optional Lab B GridSearch</td>
<td style="text-align: left;">4-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 57 · 58</td>
<td style="text-align: left;">RF vs ExtraTrees</td>
<td style="text-align: left;">5-1 · 5장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 59 ~ 62</td>
<td style="text-align: left;">Family freeze · FINAL_HOLDOUT 1회 · Closing</td>
<td style="text-align: left;">5-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 63 ~ 68</td>
<td style="text-align: left;">OOB vs CV · bootstrap=False</td>
<td style="text-align: left;">6-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 69 ~ 73</td>
<td style="text-align: left;">Clean vs Noisy</td>
<td style="text-align: left;">6-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 74 · 75</td>
<td style="text-align: left;">상황별 시나리오</td>
<td style="text-align: left;">6-3 · 6장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 76 ~ 78</td>
<td style="text-align: left;">max_features 반례</td>
<td style="text-align: left;">6-4</td>
</tr>
<tr>
<td style="text-align: left;">(새로 추가)</td>
<td style="text-align: left;">유방암으로 같은 계약</td>
<td style="text-align: left;">7. 종합 실습</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 7장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [23]:
# ---------- 1-1 ----------
X_ins, y_ins = make_classification(n_samples=260, n_features=2, n_redundant=0,   # 원본 셀 3
                                   n_informative=2, n_clusters_per_class=1, class_sep=0.9,
                                   flip_y=0.08, random_state=42)
x0 = np.linspace(X_ins[:, 0].min() - 0.7, X_ins[:, 0].max() + 0.7, 130)          # 격자 가로
x1 = np.linspace(X_ins[:, 1].min() - 0.7, X_ins[:, 1].max() + 0.7, 130)          # 격자 세로
grid_points = np.c_[[a.ravel() for a in np.meshgrid(x0, x1)]].T                  # 130 × 130 칸
tree_b, bag_b = [], []                                                           # 표본마다 경계
for seed in [3, 17, 91]:                                                         # bootstrap 3번
    idx = np.random.default_rng(seed).choice(260, size=260, replace=True)        # 복원추출
    Xb, yb = X_ins[idx], y_ins[idx]                                              # 이번 표본
    tree_b.append(DT(random_state=seed).fit(Xb, yb).predict(grid_points))        # 트리 경계
    bag = BaggingClassifier(estimator=DT(), n_estimators=80, random_state=seed)  # 80그루
    bag_b.append(bag.fit(Xb, yb).predict(grid_points))                           # Bagging 경계


def mean_pair_disagreement(bs):  # 세 경계끼리 다른 칸 비율의 평균
    return np.mean([np.mean(bs[i] != bs[j]) for i in range(3) for j in range(i + 1, 3)])  # 쌍 3개


print(f"Single Tree 경계 불일치율: {mean_pair_disagreement(tree_b):.3f}")  # 원본은 Styler로만 출력
print(f"Bagging     경계 불일치율: {mean_pair_disagreement(bag_b):.3f}")   # 흔들림 비교


# ---------- 1-2 ----------
complexity = np.linspace(0, 10, 101)              # 모델 복잡도 (원본 셀 9의 장난감 식)
bias2 = (10 - complexity) ** 2 / 10               # 복잡할수록 줄어듦
variance = complexity ** 2 / 10                   # 복잡할수록 커짐
error = bias2 + variance                          # 총 오차
best = complexity[np.argmin(error)]               # 균형점
print(f"균형점 복잡도 {best:.1f} · 총 오차 {error.min():.1f}")  # 원본 그림의 초록 점선
for c in [0, 5, 10]:                              # 세 지점
    i = int(c * 10)                               # 배열 위치
    b2, va, er = bias2[i], variance[i], error[i]  # 세 값
    print(f"복잡도 {c:>2}: bias² {b2:.1f} · variance {va:.1f} · 합 {er:.1f}")  # 표

Single Tree 경계 불일치율: 0.108
Bagging     경계 불일치율: 0.061
균형점 복잡도 5.0 · 총 오차 5.0
복잡도  0: bias² 10.0 · variance 0.0 · 합 10.0
복잡도  5: bias² 2.5 · variance 2.5 · 합 5.0
복잡도 10: bias² 0.0 · variance 10.0 · 합 10.0


In [24]:
# ---------- 2-1 ----------
wine = load_wine()                                         # 와인 178병 · 13특성 · 3품종
X_development, X_final_holdout, y_development, y_final_holdout = train_test_split(  # 먼저 잠금
    wine.data, wine.target, test_size=0.2, stratify=wine.target, random_state=42)
Xd, yd = X_development, y_development                      # 짧은 이름
FINAL_HOLDOUT_ACCESS_COUNT, FINAL_MODEL_FROZEN = 0, False  # 연 횟수 · 선택 종료 여부


def open_final_holdout_once():                                   # 원본 셀 17
    global FINAL_HOLDOUT_ACCESS_COUNT                            # 바깥 변수를 바꿈
    if not FINAL_MODEL_FROZEN:                                   # 선택이 안 끝났으면
        # 막음
        raise RuntimeError("모델과 하이퍼파라미터 선택을 먼저 완료한 뒤 FINAL_HOLDOUT을 열기")
    if FINAL_HOLDOUT_ACCESS_COUNT != 0:                          # 이미 열었으면
        raise RuntimeError("FINAL_HOLDOUT은 이미 한 번 열었음")  # 막음
    FINAL_HOLDOUT_ACCESS_COUNT += 1                              # 기록
    return X_final_holdout, y_final_holdout                      # 꺼내 줌


print("DEVELOPMENT_POOL:", Xd.shape, "· FINAL_HOLDOUT (locked):", X_final_holdout.shape)  # 142 · 36
try:                                                                 # 선택 전에 열어 보기
    open_final_holdout_once()                                        # 아직 FROZEN 아님
except RuntimeError as e:                                            # 막힘
    print("열기 시도:", e)                                           # 안내
lr5000 = LogisticRegression(max_iter=5000, random_state=42)          # 원본 설정
log_clf = Pipeline([("scaler", StandardScaler()), ("clf", lr5000)])  # 스케일러 + 로지스틱
# 이웃 7
knn_clf = Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=7))])
dt_clf = DT(max_depth=4, random_state=42)                            # 깊이 4 트리


# ---------- 2-2 ----------
members = [("lr", log_clf), ("knn", knn_clf), ("dt", dt_clf)]                   # 서로 다른 세 모델
models = {"Logistic": log_clf, "KNN": knn_clf, "Decision Tree": dt_clf,         # 단일 3개
          "Hard Voting": VotingClassifier(members, voting="hard"),              # 라벨 다수결
          "Soft Voting": VotingClassifier(members, voting="soft")}              # 확률 평균
rows, oof_predictions = [], {}                                                  # 표 · fold 밖 예측
for name, model in models.items():                                              # 같은 CV 계약
    s = cross_val_score(model, Xd, yd, cv=cv, scoring="accuracy", n_jobs=-1)    # 정확도 5개
    oof_predictions[name] = cross_val_predict(model, Xd, yd, cv=cv, n_jobs=-1)  # 2-3에서 씀
    rows.append([name, s.mean(), s.std()])                                      # 한 줄
voting_cv = pd.DataFrame(rows, columns=["Model", "CV mean", "CV std"])          # 표
voting_cv = voting_cv.sort_values("CV mean", ascending=False)                   # 1등부터
print(voting_cv.round(3).to_string(index=False))  # 원본은 Styler로만 출력

DEVELOPMENT_POOL: (142, 13) · FINAL_HOLDOUT (locked): (36, 13)
열기 시도: 모델과 하이퍼파라미터 선택을 먼저 완료한 뒤 FINAL_HOLDOUT을 열기


        Model  CV mean  CV std
     Logistic    0.979   0.028
  Hard Voting    0.972   0.040
  Soft Voting    0.965   0.031
          KNN    0.958   0.041
Decision Tree    0.902   0.059


In [25]:
# ---------- 2-3 ----------
base = ["Logistic", "KNN", "Decision Tree"]                           # 단일 모델 3개
for i, a in enumerate(base):                                          # 쌍마다
    for b in base[i + 1:]:                                            # 중복 없이
        ea, eb = oof_predictions[a] != yd, oof_predictions[b] != yd   # 각자 틀린 자리
        disagree = np.mean(oof_predictions[a] != oof_predictions[b])  # 예측이 다른 비율
        union = (ea | eb).sum()                                       # 한쪽이라도 틀린 수
        jac = (ea & eb).sum() / union if union else 0.0               # 오류 겹침
        both = np.mean(ea & eb)                                       # 둘 다 틀린 비율
        print(f"{a} vs {b}: disagreement {disagree:.3f} · "           # 쌍마다 세 지표
              f"both wrong {both:.3f} · overlap {jac:.3f}")
comparison = voting_cv.set_index("Model")["CV mean"]                  # 2-2 표
best_single = comparison.loc[base].max()                              # 단일 모델 최고
best_voting = comparison.loc[["Hard Voting", "Soft Voting"]].max()    # Voting 최고
print(f"Best single CV: {best_single:.3f} · Best voting CV: {best_voting:.3f}")  # 원본 셀 22

Logistic vs KNN: disagreement 0.028 · both wrong 0.021 · overlap 0.500
Logistic vs Decision Tree: disagreement 0.106 · both wrong 0.007 · overlap 0.062
KNN vs Decision Tree: disagreement 0.113 · both wrong 0.014 · overlap 0.111
Best single CV: 0.979 · Best voting CV: 0.972


In [26]:
# ---------- 3-1 ----------
bag_wine = BaggingClassifier(estimator=DT(random_state=42), n_estimators=300,  # 원본 셀 29
                             bootstrap=True, oob_score=True, n_jobs=-1, random_state=42)
bag_scores = cross_val_score(bag_wine, Xd, yd, cv=cv, scoring="accuracy", n_jobs=-1)  # CV
bag_wine.fit(Xd, yd)                                        # 개발 142개 전체로 fit → OOB 계산
print(f"Development CV: {bag_scores.mean():.3f} ± {bag_scores.std():.3f}")  # fold별 평균 · 흔들림
print(f"OOB estimate  : {bag_wine.oob_score_:.3f}")         # 빠진 표본의 표만 모은 추정
in_bag = np.zeros(len(yd))                                  # 샘플마다 안 빠진 횟수
for s in bag_wine.estimators_samples_:                      # 그루마다 학습에 쓴 번호
    in_bag[np.unique(s)] += 1                               # 쓴 샘플 표시
oob_rate = 1 - in_bag.mean() / 300                          # 그루당 빠진 비율
print(f"트리 한 그루에서 빠진 비율 평균: {oob_rate:.3f} (이론값 1 - 1/e ≈ 0.368)")  # 약 37%
print("FINAL_HOLDOUT access:", FINAL_HOLDOUT_ACCESS_COUNT)  # 0


# ---------- 3-2 ----------
X2, y2 = make_classification(  # 원본 셀 41 · 2차원 500개
    n_samples=500, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.0, random_state=42)
Xa, Xb, ya, yb = train_test_split(X2, y2, test_size=0.3, stratify=y2, random_state=42)  # 데모 분할


def make_bag():  # 원본 설정의 Bagging
    return BaggingClassifier(DT(), n_estimators=100, max_samples=0.8, random_state=42)  # 80%씩


tree_acc = accuracy_score(yb, DT(random_state=42).fit(Xa, ya).predict(Xb))          # 트리
bag_acc = accuracy_score(yb, make_bag().fit(Xa, ya).predict(Xb))                    # Bagging
print(f"Single Tree Demo Validation Acc: {tree_acc:.4f} · Bagging: {bag_acc:.4f}")  # 원본 셀 42
tree_cv = cross_val_score(DT(random_state=42), X2, y2, cv=cv, n_jobs=-1)            # 같은 CV
bag_cv = cross_val_score(make_bag(), X2, y2, cv=cv, n_jobs=-1)                      # 같은 CV
print(f"Single Tree Demo CV: {tree_cv.mean():.3f} ± {tree_cv.std():.3f}")           # 평균 · 흔들림
print(f"Bagging     Demo CV: {bag_cv.mean():.3f} ± {bag_cv.std():.3f}")             # 평균 · 흔들림


# ---------- 3-3 ----------
core_models = {"Single Tree": DT(random_state=42),                        # 원본 셀 44
               "Bagging": BaggingClassifier(estimator=DT(random_state=42), n_estimators=300,
                                            n_jobs=-1, random_state=42),  # 행만 무작위
               "Random Forest": RandomForestClassifier(n_estimators=300, max_features="sqrt",
                                                       # 행 + 열 무작위
                                                       n_jobs=-1, random_state=42)}
for name, model in core_models.items():                                   # 같은 CV
    s = cross_val_score(model, Xd, yd, cv=cv, scoring="accuracy", n_jobs=-1)  # 정확도 5개
    print(f"{name:<14s} CV {s.mean():.3f} ± {s.std():.3f}")               # 원본은 Styler로만 출력
for name in ["Bagging", "Random Forest"]:                                 # 트리들이 얼마나 닮았나
    m = core_models[name].fit(Xd, yd)                                     # 개발 데이터로 fit
    roots = [est.tree_.feature[0] for est in m.estimators_]               # 그루마다 첫 질문 피처
    top_share = np.bincount(roots).max() / 300                            # 가장 많은 첫 질문의 비율
    # 닮은 정도
    print(f"{name}: 첫 질문 피처 종류 {len(set(roots))}개 · 최다 피처 비율 {top_share:.2f}")

Development CV: 0.944 ± 0.041
OOB estimate  : 0.944
트리 한 그루에서 빠진 비율 평균: 0.366 (이론값 1 - 1/e ≈ 0.368)
FINAL_HOLDOUT access: 0


Single Tree Demo Validation Acc: 0.9400 · Bagging: 0.9667


Single Tree Demo CV: 0.944 ± 0.022
Bagging     Demo CV: 0.952 ± 0.021
Single Tree    CV 0.902 ± 0.059


Bagging        CV 0.944 ± 0.041


Random Forest  CV 0.979 ± 0.028
Bagging: 첫 질문 피처 종류 6개 · 최다 피처 비율 0.63


Random Forest: 첫 질문 피처 종류 11개 · 최다 피처 비율 0.21


In [27]:
from sklearn.inspection import permutation_importance  # 섞어서 성능 변화 보기
from sklearn.model_selection import GridSearchCV       # 하이퍼파라미터 탐색


# ---------- 4-1 ----------

X_work, X_feat_eval, y_work, y_feat_eval = train_test_split(        # 25% untouched evaluation
    Xd, yd, test_size=0.25, stratify=yd, random_state=42)
X_feat_fit, X_feat_sel, y_feat_fit, y_feat_sel = train_test_split(  # 나머지를 fit · selection
    X_work, y_work, test_size=1 / 3, stratify=y_work, random_state=43)
sizes = [len(y_feat_fit), len(y_feat_sel), len(y_feat_eval)]        # 세 역할 크기
print("fit / selection / untouched evaluation =", " / ".join(map(str, sizes)))  # 70 / 36 / 36
RF300 = dict(n_estimators=300, max_features="sqrt", n_jobs=-1, random_state=42)  # 이 장의 RF 설정
rf_fit = RandomForestClassifier(**RF300)                            # 중요도 계산용
rf_fit.fit(X_feat_fit, y_feat_fit)                                  # fit 70개로만
base_ok = int((rf_fit.predict(X_feat_sel) == y_feat_sel).sum())     # selection 정답 수
print(f"Selection baseline: {base_ok}/{len(y_feat_sel)} = {base_ok / len(y_feat_sel):.4f}")  # 천장
# selection에서만
feature_perm = permutation_importance(rf_fit, X_feat_sel, y_feat_sel, n_repeats=20,
                                      scoring="accuracy", n_jobs=-1, random_state=42)
perm = pd.Series(feature_perm.importances_mean, index=wine.feature_names)  # 피처별 평균 감소
perm = perm.sort_values(ascending=False)                            # 큰 순서
print(perm.head(6).round(6).to_string())                            # 위 6개
print("0인 피처 수:", int((perm == 0).sum()), "/ 13")               # 감소가 관찰되지 않은 피처


# ---------- 4-2 ----------
important_mask = feature_perm.importances_mean > 0.0  # selection에서 고정한 mask
print("고른 피처:", np.array(wine.feature_names)[important_mask].tolist())         # 5개
X_refit = np.concatenate([X_feat_fit, X_feat_sel])                                 # fit + selection
y_refit = np.concatenate([y_feat_fit, y_feat_sel])                                 # 같은 학습 풀
rf_all, rf_sel = RandomForestClassifier(**RF300), RandomForestClassifier(**RF300)  # 같은 설정 두 개
ok_all = int((rf_all.fit(X_refit, y_refit).predict(X_feat_eval) == y_feat_eval).sum())  # 전체 피처
rf_sel.fit(X_refit[:, important_mask], y_refit)                                    # 고른 피처만
ok_sel = int((rf_sel.predict(X_feat_eval[:, important_mask]) == y_feat_eval).sum())  # 고른 피처
print(f"All features: {ok_all}/36 = {ok_all / 36:.4f} · "                          # 전체 피처
      f"Selected features: {ok_sel}/36 = {ok_sel / 36:.4f}")
print(f"Observed difference (selected - all): {(ok_sel - ok_all) / 36:+.4f}")      # 1개 = 0.0278


# ---------- 4-3 ----------

corr = pd.DataFrame(X_feat_fit, columns=wine.feature_names).corr()           # fit 부분만 기술
names = wine.feature_names                                                   # 짧은 이름
for i in range(13):                                                          # 모든 쌍
    for j in range(i + 1, 13):                                               # 중복 없이
        if abs(corr.iloc[i, j]) > 0.7:                                       # 강한 상관
            print(f"  {names[i]} ↔ {names[j]}  r = {corr.iloc[i, j]:+.3f}")  # 원본 셀 51
grid = {"n_estimators": [100, 200, 400], "max_depth": [None, 10, 20, 30],    # 3 × 4 × 3 = 36
        "max_features": ["sqrt", "log2", 0.5]}
gs_rf = GridSearchCV(RandomForestClassifier(random_state=42), grid,  # 원본 셀 56 · 조합 단위로 병렬
                     scoring="accuracy", cv=cv, n_jobs=-1).fit(Xd, yd)
top = pd.DataFrame(gs_rf.cv_results_)["mean_test_score"]                     # 36조합 점수
print("best_params_:", gs_rf.best_params_, f"· best_score_ {gs_rf.best_score_:.4f}")  # 순서상 먼저
print("최고 점수 동점 조합 수:", int((top == top.max()).sum()), "/ 36")      # 유일한 정답이 아님

fit / selection / untouched evaluation = 70 / 36 / 36


Selection baseline: 36/36 = 1.0000


color_intensity    0.043056
proline            0.027778
alcohol            0.026389
hue                0.019444
flavanoids         0.009722
magnesium          0.000000
0인 피처 수: 8 / 13
고른 피처: ['alcohol', 'flavanoids', 'color_intensity', 'hue', 'proline']


All features: 36/36 = 1.0000 · Selected features: 35/36 = 0.9722
Observed difference (selected - all): -0.0278
  total_phenols ↔ flavanoids  r = +0.884
  total_phenols ↔ od280/od315_of_diluted_wines  r = +0.748
  flavanoids ↔ od280/od315_of_diluted_wines  r = +0.797


best_params_: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 100} · best_score_ 0.9791
최고 점수 동점 조합 수: 28 / 36


In [28]:
from sklearn.base import clone                     # 설정만 복사한 새 모델
from sklearn.metrics import classification_report  # 클래스별 표


# ---------- 5-1 ----------
rf_candidate = RandomForestClassifier(**RF300)                                # 4장과 같은 RF 설정
# 기준값까지 무작위
et_candidate = ExtraTreesClassifier(n_estimators=300, max_features="sqrt", bootstrap=False,
                                    n_jobs=-1, random_state=42)
family_models = {"Random Forest": rf_candidate, "ExtraTrees": et_candidate}   # 공식 후보 둘
rows = []                                                                     # family별 점수
for name, model in family_models.items():                                     # 같은 CV 계약
    s = cross_val_score(model, Xd, yd, cv=cv, scoring="accuracy", n_jobs=-1)  # 정확도 5개
    rows.append([name, s.mean(), s.std()])                                    # 한 줄
extra_evidence = pd.DataFrame(rows, columns=["Model", "CV mean", "CV std"])   # 표
extra_evidence = extra_evidence.sort_values("CV mean", ascending=False)       # 1등부터
print(extra_evidence.round(3).to_string(index=False))  # 원본은 Styler로만 출력


# ---------- 5-2 ----------

family_scores = extra_evidence.set_index("Model")["CV mean"]      # 5-1 표
selected = family_scores.idxmax()                                 # 개발 CV 1위
sel_cv = family_scores[selected]                                  # 선택 근거 점수
print(f"선택된 family: {selected} · 근거 DEVELOPMENT_POOL CV mean {sel_cv:.3f}")  # 고정
FINAL_MODEL_FROZEN = True                                         # 선택 종료
X_final_once, y_final_once = open_final_holdout_once()            # 정확히 한 번
final_model = clone(family_models[selected]).fit(Xd, yd)          # 개발 142개로 학습
y_pred_final = final_model.predict(X_final_once)                  # 36개 예측
print("FINAL_HOLDOUT access count:", FINAL_HOLDOUT_ACCESS_COUNT)  # 1
print("FINAL_HOLDOUT Accuracy:", round(accuracy_score(y_final_once, y_pred_final), 3))  # 보고
tn_ = wine.target_names                                           # 품종 이름
report = classification_report(y_final_once, y_pred_final, target_names=tn_, digits=3)  # 표
print(report)                                                     # 클래스별 정밀도 · 재현율

        Model  CV mean  CV std
   ExtraTrees    0.986   0.017
Random Forest    0.979   0.028
선택된 family: ExtraTrees · 근거 DEVELOPMENT_POOL CV mean 0.986


FINAL_HOLDOUT access count: 1
FINAL_HOLDOUT Accuracy: 1.0
              precision    recall  f1-score   support

     class_0      1.000     1.000     1.000        12
     class_1      1.000     1.000     1.000        14
     class_2      1.000     1.000     1.000        10

    accuracy                          1.000        36
   macro avg      1.000     1.000     1.000        36
weighted avg      1.000     1.000     1.000        36



In [29]:
# ---------- 6-1 ----------
try:  # 빠지는 샘플이 없는데 OOB를 요청
    # 오류 예상
    RandomForestClassifier(n_estimators=10, bootstrap=False, oob_score=True).fit(Xd, yd)
except ValueError as e:                                                # sklearn이 막음
    print("OOB 요청 결과:", str(e)[:58])                               # 앞부분만
# 표본 무작위 없음
rf_nb = RandomForestClassifier(n_estimators=300, bootstrap=False, max_features="sqrt",
                               n_jobs=-1, random_state=42).fit(Xd, yd)
roots = [est.tree_.feature[0] for est in rf_nb.estimators_]            # 그루마다 첫 질문
print("bootstrap=False 첫 질문 피처 종류:", len(set(roots)), "개")     # 피처 무작위는 남음
print("ExtraTrees 기본 bootstrap:", ExtraTreesClassifier().bootstrap)  # False


# ---------- 6-2 ----------
def practical_rf_et_label(rf_mean, et_mean, threshold=0.005):  # 표시용 규칙 (유의성 검정 아님)
    diff = abs(rf_mean - et_mean)                              # 평균 차이
    if diff < threshold:                                       # 아주 작으면
        return "비슷 (명확한 승자 주장 어려움)", diff          # 판정 보류
    return ("RF" if rf_mean > et_mean else "ET"), diff         # 높은 쪽


clean = make_classification(n_samples=5000, n_features=5, n_informative=5,   # 경계 뚜렷 · 노이즈 0
                            n_redundant=0, class_sep=3.0, flip_y=0.0, random_state=42)
noisy = make_classification(n_samples=5000, n_features=50, n_informative=5,  # 노이즈 많음
                            n_redundant=10, class_sep=0.3, flip_y=0.3, random_state=42)
data = {"Clean": clean, "Noisy": noisy}                                      # 두 실험
for name, (Xs, ys) in data.items():                                          # 두 실험
    # RF
    rf = cross_val_score(RandomForestClassifier(200, random_state=42), Xs, ys, cv=cv, n_jobs=-1)
    et = cross_val_score(ExtraTreesClassifier(200, random_state=42), Xs, ys, cv=cv, n_jobs=-1)  # ET
    label, diff = practical_rf_et_label(rf.mean(), et.mean())                # 표시
    # 평균 ± 흔들림
    rf_t, et_t = f"{rf.mean():.4f} ± {rf.std():.4f}", f"{et.mean():.4f} ± {et.std():.4f}"
    print(f"{name}: RF {rf_t} · ET {et_t} · 차이 {diff:.4f} → {label}")      # 한 줄


# ---------- 6-3 ----------
scenarios = {"경계 뚜렷 + feature 적음": (10, 8, 0.0, 2.0),    # (특성 수, 유용, 라벨 노이즈, 분리)
             "노이즈 많음 + feature 많음": (50, 5, 0.3, 0.3),
             "노이즈 많음 + feature 적음": (10, 5, 0.3, 0.5),
             "경계 뚜렷 + feature 많음(노이즈)": (50, 5, 0.0, 2.0)}
# 원본 셀 75 (특성 수 · 유용 · 라벨 노이즈 · 분리)
for name, (nf, ni, flip, sep) in scenarios.items():
    Xs, ys = make_classification(                              # 시나리오 데이터 2000개
        n_samples=2000, n_features=nf, n_informative=ni, n_redundant=0,
        flip_y=flip, class_sep=sep, random_state=42)
    # RF
    rf = cross_val_score(RandomForestClassifier(100, random_state=42), Xs, ys, cv=cv, n_jobs=-1)
    et_m = ExtraTreesClassifier(100, bootstrap=False, random_state=42)  # ET
    et = cross_val_score(et_m, Xs, ys, cv=cv, n_jobs=-1)       # ET
    label, diff = practical_rf_et_label(rf.mean(), et.mean())  # 6-2 규칙
    print(f"{name}: RF {rf.mean():.4f} · ET {et.mean():.4f} · 차이 {diff:.4f} → {label}")  # 한 줄

OOB 요청 결과: Out of bag estimation only available if bootstrap=True


bootstrap=False 첫 질문 피처 종류: 11 개
ExtraTrees 기본 bootstrap: False


Clean: RF 0.9990 ± 0.0009 · ET 0.9996 ± 0.0005 · 차이 0.0006 → 비슷 (명확한 승자 주장 어려움)


Noisy: RF 0.7116 ± 0.0119 · ET 0.7122 ± 0.0172 · 차이 0.0006 → 비슷 (명확한 승자 주장 어려움)


경계 뚜렷 + feature 적음: RF 0.9590 · ET 0.9695 · 차이 0.0105 → ET


노이즈 많음 + feature 많음: RF 0.6385 · ET 0.6105 · 차이 0.0280 → RF


노이즈 많음 + feature 적음: RF 0.7330 · ET 0.7270 · 차이 0.0060 → RF


경계 뚜렷 + feature 많음(노이즈): RF 0.9735 · ET 0.9780 · 차이 0.0045 → 비슷 (명확한 승자 주장 어려움)


In [30]:
# ---------- 6-4 ----------
X_many, y_many = make_classification(n_samples=300, n_features=200, n_informative=5,  # 원본 셀 77
                                     n_redundant=0, flip_y=0.1, class_sep=0.3, random_state=42)
# 후보 5개
opts = {"sqrt (14개)": "sqrt", "log2 (7개)": "log2", "5개": 5, "50개": 50, "None (200개)": None}
rows = []                                                                # 설정별 점수
for name, mf in opts.items():                                            # 설정마다
    rf = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=42)  # 같은 RF
    s = cross_val_score(rf, X_many, y_many, cv=cv, scoring="accuracy", n_jobs=-1)  # 5-Fold 병렬
    rows.append([name, s.mean(), s.std()])                               # 한 줄
res = pd.DataFrame(rows, columns=["max_features", "CV mean", "CV std"])  # 표
print(res.round(4).to_string(index=False))                               # 원본 표
chance = np.bincount(y_many).max() / len(y_many)                         # 다수 클래스 비율
best = res.loc[res["CV mean"].idxmax()]                                  # 최고 행
n_below = int((res["CV mean"] < chance).sum())                           # chance 미만 설정 수
gap = best["CV mean"] - chance                                           # 최고와 chance 차이
print(f"chance {chance:.3f} · chance 미만 {n_below}/5개 · "              # 근거 판단
      f"최고와 chance 차이 {gap:+.4f} (std {best['CV std']:.4f})")

max_features  CV mean  CV std
  sqrt (14개)   0.5400  0.0389
   log2 (7개)   0.4900  0.0688
          5개   0.5200  0.1008
         50개   0.4867  0.1108
 None (200개)   0.4600  0.0712
chance 0.530 · chance 미만 4/5개 · 최고와 chance 차이 +0.0100 (std 0.0389)


In [31]:
# ---------- 7. 종합 실습 ----------
from sklearn.datasets import load_breast_cancer  # 유방암 569건 · 30특성


# 1단계: 새 데이터로 새 계약 (Wine holdout은 이미 소비)

bc = load_breast_cancer()                                                 # 0 = 악성, 1 = 양성
Xbd, Xbh, ybd, ybh = train_test_split(bc.data, bc.target, test_size=0.2,  # 잠금 20%
                                      stratify=bc.target, random_state=42)
print("개발", Xbd.shape, "· 잠금", Xbh.shape)                             # 455 · 114


# 2 · 3단계: 후보 5개를 같은 CV로 → 1위 고정
lr_bc = Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=5000))])  # 투표 구성원
cands = {"Single Tree": DT(random_state=42),                                # 기준
         "Bagging": BaggingClassifier(DT(), n_estimators=200, n_jobs=-1, random_state=42),
         "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
         "ExtraTrees": ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42),
         "Soft Voting": VotingClassifier([("lr", lr_bc),                    # 서로 다른 셋
                                          ("rf", RandomForestClassifier(300, random_state=42)),
                                          ("dt", DT(max_depth=4, random_state=42))],
                                         voting="soft")}
scores = {}                                                                 # 후보별 CV 평균
for name, m in cands.items():                                               # 같은 계약
    s = cross_val_score(m, Xbd, ybd, cv=cv, scoring="accuracy", n_jobs=-1)  # 정확도 5개
    scores[name] = s.mean()                                                 # 저장
    print(f"{name:<14s} CV {s.mean():.4f} ± {s.std():.4f}")                 # 표
pick = max(scores, key=scores.get)                                          # 개발 CV 1위
print("고정:", pick)                                                        # 선택 종료


# 4단계: holdout 1회 (고정한 후보 그대로)
final_bc = clone(cands[pick]).fit(Xbd, ybd)                       # 개발 455개로 학습
acc_h = accuracy_score(ybh, final_bc.predict(Xbh))                # 잠금 114개 · 처음이자 마지막
print(f"holdout 114개 정확도: {acc_h:.4f} · 개발 CV {scores[pick]:.4f}")  # 보고
print("틀린 건수:", int(round((1 - acc_h) * len(ybh))), "/ 114")  # 1건 = 0.0088

개발 (455, 30) · 잠금 (114, 30)
Single Tree    CV 0.9165 ± 0.0179


Bagging        CV 0.9516 ± 0.0215


Random Forest  CV 0.9648 ± 0.0176


ExtraTrees     CV 0.9736 ± 0.0164


Soft Voting    CV 0.9538 ± 0.0224
고정: ExtraTrees


holdout 114개 정확도: 0.9561 · 개발 CV 0.9736
틀린 건수: 5 / 114
